In [1]:
#!/usr/bin/env python3
"""Local Linux smoke runs using the repository's gcp_setup_stellar_private.sh.

Edit the settings below, then run: python3 run_local_protocols.py
Requires the repository's build dependencies. No GCP, commit or push operations.
All replicas share one host, so these are functional tests, not distributed benchmarks.
"""

import csv
import fcntl
import hashlib
import json
import os
from pathlib import Path
import re
import shlex
import signal
import socket
import subprocess
import tempfile
import time
from datetime import datetime
from urllib.request import build_opener, ProxyHandler


# ---------------- Settings: edit these ----------------
REPO = Path.home() / "stellar-core"
RESULTS_ROOT = Path.home() / "stellar-local-results"
NUM_NODES = 4                 # Start with 4; use 8 to match your GCP script.
PROTOCOLS = ["shabdiz", "iths", "pbft", "tetrabft", "simple_it", "forget_it", "scp"]
# Add "simple_it_bracha" to exercise Simple-IT's non-optimistic variant.
BUILD_JOBS = 8
REGENERATE_BUILD = False       # Needed on first run after the source-file split.
CONFIGURE_ARGS = None         # None reuses ./config.status --config, if present.
# For a fresh checkout, set your normal options, e.g. CONFIGURE_ARGS = [].
RESTORE_ORIGINAL_BUILD = False  # Restore header AND rebuild original mode afterward.
CXX = os.environ.get("CXX", "g++")

WORKLOAD = "WORKLOAD_W"       # Write-only exercises consensus for every request.
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 60
CLIENT_DRAIN_TIMEOUT_SEC = 15
STARTUP_TIMEOUT_SEC = 120
SEND_INTERVAL_US = 0
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 1, "max_in_flight": 150},
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]
SCP_OBSERVATION_SEC = 60       # Native diagnostic; no shab_client for SCP.
# ------------------------------------------------------

MODE_FLAGS = {
    "iths": "ITHS_MODE", "pbft": "PBFT_MODE", "scp": "ENABLE_SCP_TRACKING",
    "tetrabft": "TETRABFT_MODE", "simple_it": "SIMPLE_IT_MODE",
    "forget_it": "FORGET_IT_MODE", "simple_it_bracha": "SIMPLE_IT_MODE",
}
COMMIT_TAGS = {
    "shabdiz": ("[FAST COMMITTED]",),
    # Accept both the legacy and native blog adapter's commit labels.
    "iths": ("[IT-HS COMMITTED]", "[IT-HS-BLOG COMMITTED]"),
    "pbft": ("[PBFT COMMITTED-LOCAL]",), "tetrabft": ("[TETRA COMMITTED]",),
    "simple_it": ("[SIMPLE-IT COMMITTED]",), "simple_it_bracha": ("[SIMPLE-IT COMMITTED]",),
    "forget_it": ("[FORGET-IT COMMITTED]",),
}
HTTP = build_opener(ProxyHandler({}))  # Local endpoints must bypass HTTP proxies.


def save_json(path, value):
    path.write_text(json.dumps(value, indent=2) + "\n")


def replace_once(text, pattern, replacement):
    result, count = re.subn(pattern, lambda _: replacement, text, flags=re.MULTILINE)
    if count != 1:
        raise RuntimeError(f"Expected one match for {pattern!r}, found {count}; check repository changes")
    return result


def set_constant(text, name, value):
    prefix = re.search(rf"^inline constexpr [^\n=]+\b{re.escape(name)}\s*=", text,
                       re.MULTILINE)
    if prefix is None:
        raise RuntimeError(f"Missing experiment setting: {name}")
    return replace_once(text, rf"^inline constexpr [^\n=]+\b{re.escape(name)}\s*=\s*[^;]+;",
                        prefix.group(0) + " " + str(value) + ";")


def select_protocol(original, protocol):
    text = original
    for flag in set(MODE_FLAGS.values()):
        text = set_constant(text, flag, "true" if MODE_FLAGS.get(protocol) == flag else "false")
    for name, value in {
        "SIMPLE_IT_OPTIMISTIC": "false" if protocol == "simple_it_bracha" else "true",
        "NonPrune_MODE": "false", "currentWorkload": WORKLOAD,
        "SERVER_BATCH_SIZE": SERVER_BATCH_SIZE, "CLIENT_PORT": CLIENT_PORT,
    }.items():
        text = set_constant(text, name, value)
    return text


def stop_processes(processes):
    # Only process groups started by THIS script; never pkill other experiments.
    for proc in processes:
        if proc.poll() is None:
            try:
                os.killpg(proc.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
    deadline = time.monotonic() + 5
    for proc in processes:
        try:
            proc.wait(timeout=max(0.01, deadline - time.monotonic()))
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            proc.wait()


def command(args, cwd, log):
    with log.open("a") as out:
        out.write("\n$ " + shlex.join(map(str, args)) + "\n")
        out.flush()
        proc = subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)
        try:
            code = proc.wait()
            if code:
                raise RuntimeError(f"Command failed ({code}); see {log}")
        finally:
            stop_processes([proc])


def launch(args, cwd, log):
    with log.open("w") as out:
        return subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)


def ensure_alive(processes):
    for i, proc in enumerate(processes, 1):
        if proc.poll() is not None:
            raise RuntimeError(f"Replica {i} exited with code {proc.returncode}; inspect its logs")


def check_ports(ports):
    if len(set(ports)) != len(ports):
        raise RuntimeError("Peer, HTTP, and client ports overlap")
    for port in ports:
        with socket.socket() as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("0.0.0.0", port))
            except OSError as exc:
                raise RuntimeError(f"Port {port} is occupied; stop the previous local run") from exc


def local_setup_text(source, root):
    # Reuse your generator, including its seeds, validator lists and port arithmetic.
    text = replace_once(source, r"^STELLAR_CORE=.*$", "STELLAR_CORE=" + shlex.quote(str(REPO / "src/stellar-core")))
    text = replace_once(text, r"^BASE_DIR=.*$", "BASE_DIR=" + shlex.quote(str(root / "nodes")))
    text = replace_once(text, r"^mapfile -t NODE_IPS <.*$",
                        "mapfile -t NODE_IPS < " + shlex.quote(str(root / "localhost_ips.txt")))
    return text.replace('$STELLAR_CORE gen-seed', '"$STELLAR_CORE" gen-seed')


def prepare_nodes(root, setup_source):
    root.mkdir(parents=True)
    (root / "localhost_ips.txt").write_text("127.0.0.1\n" * NUM_NODES)
    setup = root / "setup-local.sh"
    setup.write_text(local_setup_text(setup_source, root))
    command(["bash", setup, "start"], root, root / "setup.log")
    configs, ports = [], []
    for i in range(1, NUM_NODES + 1):
        cfg = root / "nodes" / f"node{i}" / "stellar-core.cfg"
        text = cfg.read_text()
        # All added settings must precede the generator's TOML table headers.
        additions = {
            "SEND_CUSTOM_MESSAGE": "true" if i == 1 else "false",
            "ALLOW_LOCALHOST_FOR_TESTING": "true", "PUBLIC_HTTP_PORT": "false",
            "TARGET_PEER_CONNECTIONS": str(NUM_NODES - 1),
            "MAX_ADDITIONAL_PEER_CONNECTIONS": str(8 * NUM_NODES),
        }
        for key, value in additions.items():
            if re.search(rf"^{key}\s*=", text, re.MULTILINE):
                text = replace_once(text, rf"^{key}\s*=.*$", f"{key}={value}")
            else:
                text = f"{key}={value}\n" + text
        cfg.write_text(text)
        peer = int(re.search(r"^PEER_PORT=(\d+)", text, re.MULTILINE)[1])
        http = int(re.search(r"^HTTP_PORT=(\d+)", text, re.MULTILINE)[1])
        ports.extend([peer, http])
        configs.append((cfg, http))
    check_ports(ports + [CLIENT_PORT])
    # The generator's init branch continues on new-db failure. Check each command here.
    for cfg, _ in configs:
        command([REPO / "src/stellar-core", "new-db", "--conf", cfg], cfg.parent,
                cfg.parent / "init.log")
    return configs


def wait_for_network(replicas, configs, custom):
    deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
    while time.monotonic() < deadline:
        ensure_alive(replicas)
        logs = [(cfg.parent / "stellar-core.log").read_text(errors="replace")
                if (cfg.parent / "stellar-core.log").exists() else "" for cfg, _ in configs]
        if all("Shabdiz started" in log for log in logs):
            if not custom or f"Client listener started on port {CLIENT_PORT}" in logs[0]:
                return
        time.sleep(1)
    raise TimeoutError("Full peer connectivity/client readiness timed out; inspect node logs")


def info_snapshot(configs):
    results = []
    for _, port in configs:
        try:
            with HTTP.open(f"http://127.0.0.1:{port}/info", timeout=3) as response:
                results.append(json.load(response))
        except Exception as exc:
            results.append({"error": str(exc)})
    return results


def count_commits(path, protocol):
    text = path.read_text(errors="replace")
    return sum(text.count(tag) for tag in COMMIT_TAGS[protocol])


def client_metrics(paths):
    count, latency_sum = 0, 0.0
    for path in paths:
        for line in path.read_text(errors="replace").splitlines():
            if "[CLIENT_SEC]" not in line:
                continue
            fields = dict(re.findall(r"(\w+)=([^\s]+)", line))
            if 1 <= int(fields["sec"]) <= CLIENT_DURATION_SEC:
                n = int(fields["requests"])
                count += n
                if n:
                    latency_sum += n * float(fields["avg_latency_ms"])
    return {"acked_in_window": count, "throughput_ops_sec": count / CLIENT_DURATION_SEC,
            "mean_latency_ms": latency_sum / count if count else None}


def run_trial(protocol, root, setup_source, load):
    replicas, clients = [], []
    result = {"protocol": protocol, **(load or {}), "status": "failed"}
    try:
        configs = prepare_nodes(root, setup_source)
        save_json(root / "run_config.json", {
            **result, "num_nodes": NUM_NODES, "workload": WORKLOAD,
            "batch_size": SERVER_BATCH_SIZE, "client_duration_sec": CLIENT_DURATION_SEC,
            "client_port": CLIENT_PORT, "send_interval_us": SEND_INTERVAL_US,
            "aggregate_max_in_flight": (load["active_clients"] * load["client_threads"] *
                                        load["max_in_flight"]) if load else None,
        })
        for cfg, _ in configs:
            replicas.append(launch([REPO / "src/stellar-core", "run", "--conf", cfg],
                                   cfg.parent, cfg.parent / "console.log"))
        wait_for_network(replicas, configs, protocol != "scp")
        if protocol == "scp":
            save_json(root / "info-before.json", info_snapshot(configs))
            end = time.monotonic() + SCP_OBSERVATION_SEC
            while time.monotonic() < end:
                ensure_alive(replicas)
                time.sleep(1)
            save_json(root / "info-after.json", info_snapshot(configs))
            result.update(status="diagnostic_only", note=(
                "Native SCP workload creates accounts locally. Ledger advancement alone "
                "does not establish successful workload application; inspect node logs. "
                "Do not compare this run with custom-client throughput."))
        else:
            paths = []
            for i in range(load["active_clients"]):
                path = root / f"client-{i + 1}.log"
                paths.append(path)
                clients.append(launch([
                    root.parent / "shab_client", "127.0.0.1", CLIENT_PORT,
                    load["max_in_flight"], 100000000, SERVER_BATCH_SIZE,
                    CLIENT_DURATION_SEC, SEND_INTERVAL_US, load["client_threads"],
                ], root, path))
            end = time.monotonic() + CLIENT_DURATION_SEC + CLIENT_DRAIN_TIMEOUT_SEC
            while any(p.poll() is None for p in clients) and time.monotonic() < end:
                ensure_alive(replicas)
                time.sleep(0.5)
            ensure_alive(replicas)
            timed_out = any(p.poll() is None for p in clients)
            codes = [p.poll() for p in clients]
            stop_processes(clients)
            metrics = client_metrics(paths)
            per_client = [client_metrics([path])["acked_in_window"] for path in paths]
            commits = [count_commits(cfg.parent / "stellar-core.log", protocol)
                       for cfg, _ in configs]
            result.update(metrics, client_exit_codes=codes, per_client_acked=per_client,
                          per_replica_commit_log_counts=commits)
            if (any(code not in (None, 0) for code in codes) or not all(per_client) or
                    (WORKLOAD != "WORKLOAD_R100" and not all(commits))):
                raise RuntimeError("A client failed, a client received no ACKs, or a replica logged no commits")
            result["status"] = "drain_timeout" if timed_out else "ok"
            if timed_out:
                result["note"] = ("Client did not finish draining within the grace period. "
                                  "A final partial batch can cause this; inspect logs for stalls.")
    except Exception as exc:
        result.update(status="failed", error=str(exc))
    finally:
        stop_processes(clients)
        stop_processes(replicas)
        root.mkdir(parents=True, exist_ok=True)
        save_json(root / "result.json", result)
    return result


def write_summary(session, results):
    save_json(session / "summary.json", results)
    fields = ["protocol", "active_clients", "client_threads", "max_in_flight", "status",
              "acked_in_window", "throughput_ops_sec", "mean_latency_ms", "error", "note"]
    with (session / "summary.csv").open("w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)


def main():
    global REPO, RESULTS_ROOT
    REPO, RESULTS_ROOT = REPO.expanduser().resolve(), RESULTS_ROOT.expanduser().resolve()
    header = REPO / "src/overlay/experiments/ExperimentConfig.h"
    source = REPO / "gcp_setup_stellar_private.sh"
    if not header.is_file() or not source.is_file():
        raise RuntimeError("Set REPO to your updated stellar-core checkout")
    if NUM_NODES < 4 or not PROTOCOLS or not LOAD_POINTS:
        raise ValueError("Use at least four replicas and nonempty protocol/load lists")
    if CLIENT_DURATION_SEC <= 0 or SERVER_BATCH_SIZE <= 0 or BUILD_JOBS <= 0:
        raise ValueError("Duration, batch size and build jobs must be positive")
    if not re.fullmatch(r"WORKLOAD_(A|B|F|W|R0|R10|R20|R25|R40|R50|R60|R75|R80|R100)", WORKLOAD):
        raise ValueError("Unknown WORKLOAD")
    for protocol in PROTOCOLS:
        if protocol != "shabdiz" and protocol not in MODE_FLAGS:
            raise ValueError(f"Unknown protocol: {protocol}")
    for load in LOAD_POINTS:
        if min(load.values()) <= 0:
            raise ValueError("Client counts, threads and in-flight limits must be positive")
        if load["active_clients"] * load["client_threads"] * load["max_in_flight"] < SERVER_BATCH_SIZE:
            raise ValueError("Aggregate in-flight limit must cover at least one server batch")
    # Separate invocations must not race while changing this checkout's build flags.
    lockname = "stellar-local-" + hashlib.sha256(str(REPO).encode()).hexdigest()[:16] + ".lock"
    with (Path(tempfile.gettempdir()) / lockname).open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        session = RESULTS_ROOT / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        session.mkdir(parents=True)
        original = header.read_bytes()
        setup_source = source.read_text()
        (session / "ExperimentConfig.original.h").write_bytes(original)
        (session / "setup.original.sh").write_text(setup_source)
        command(["git", "rev-parse", "HEAD"], REPO, session / "source.log")
        command(["git", "status", "--short"], REPO, session / "source.log")
        command(["git", "diff", "HEAD"], REPO, session / "source.diff")
        print(f"Results: {session}", flush=True)
        results, changed, interrupted = [], False, False
        try:
            if REGENERATE_BUILD:
                options = CONFIGURE_ARGS
                if options is None:
                    if not (REPO / "config.status").exists():
                        raise RuntimeError("Fresh checkout: set CONFIGURE_ARGS to your configure options")
                    options = shlex.split(subprocess.check_output(
                        ["./config.status", "--config"], cwd=REPO, text=True))
                command(["./autogen.sh"], REPO, session / "bootstrap.log")
                command(["./configure", *options], REPO, session / "bootstrap.log")
            for protocol in PROTOCOLS:
                print(f"\nBuilding {protocol} ...", flush=True)
                folder = session / protocol
                folder.mkdir()
                selected = select_protocol(original.decode(), protocol)
                changed = True
                header.write_text(selected)
                (folder / "ExperimentConfig.h").write_text(selected)
                command(["make", f"-j{BUILD_JOBS}"], REPO, folder / "build.log")
                if protocol != "scp":
                    command([*shlex.split(CXX), "-O2", "-std=c++17", "-pthread",
                             "-I" + str(REPO / "src"), REPO / "shab_client.cpp",
                             "-o", folder / "shab_client"], REPO, folder / "build.log")
                for index, load in enumerate([None] if protocol == "scp" else LOAD_POINTS):
                    label = "native" if load is None else (
                        f"load{index + 1}_c{load['active_clients']}_t{load['client_threads']}_w{load['max_in_flight']}")
                    print(f"Running {protocol}/{label} ...", flush=True)
                    result = run_trial(protocol, folder / label, setup_source, load)
                    results.append(result)
                    write_summary(session, results)
                    print(json.dumps(result), flush=True)
        except KeyboardInterrupt:
            interrupted = True
            print("Interrupted; stopping this script's processes and restoring settings.", flush=True)
            raise
        finally:
            if changed:
                header.write_bytes(original)
                if RESTORE_ORIGINAL_BUILD and not interrupted:
                    print("Restoring original build ...", flush=True)
                    command(["make", f"-j{BUILD_JOBS}"], REPO, session / "restore-build.log")
                else:
                    print("Header restored. Run make before using src/stellar-core manually; "
                          "its binary may still use the last tested protocol.", flush=True)
        print(f"\nFinished. Summary: {session / 'summary.csv'}", flush=True)
        return 1 if any(r["status"] == "failed" for r in results) else 0


def interrupted_by_signal(*_):
    raise KeyboardInterrupt


if __name__ == "__main__":
    signal.signal(signal.SIGTERM, interrupted_by_signal)
    raise SystemExit(main())


Results: /home/tejas/stellar-local-results/20261105-183009-349449

Building shabdiz ...
Running shabdiz/load1_c1_t1_w150 ...
{"protocol": "shabdiz", "active_clients": 1, "client_threads": 1, "max_in_flight": 150, "status": "drain_timeout", "acked_in_window": 140300, "throughput_ops_sec": 2338.3333333333335, "mean_latency_ms": 64.02366856735567, "client_exit_codes": [null], "per_client_acked": [140300], "per_replica_commit_log_counts": [1404, 1404, 1404, 1404], "note": "Client did not finish draining within the grace period. A final partial batch can cause this; inspect logs for stalls."}
Running shabdiz/load2_c1_t2_w150 ...
{"protocol": "shabdiz", "active_clients": 1, "client_threads": 2, "max_in_flight": 150, "status": "drain_timeout", "acked_in_window": 2990600, "throughput_ops_sec": 49843.333333333336, "mean_latency_ms": 5.988167683408013, "client_exit_codes": [null], "per_client_acked": [2990600], "per_replica_commit_log_counts": [29909, 29909, 29909, 29909], "note": "Client did no

SystemExit: 0

/home/tejas/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# GCP Client Load: Throughput vs Latency (All Protocols)

In [11]:
"""GCP throughput versus ACK latency; run this cell after editing the settings.
Builds and runs only when this cell is executed. No automatic git commit/push.
Checks the image's configured build once; all protocols/sizes reuse the binary.
Creates/prepares VMs in parallel, then sweeps protocols and loads on the same fleet.
Pins one source commit and preserves raw logs.
"""
import concurrent.futures
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import socket
import subprocess
import tarfile
import tempfile
import threading
import time
import uuid

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Your local updated stellar-core checkout.
RESULTS_DIR = Path.home() / "work/experiments/gcp-client-load/ClientThputLatency"
# Fixed output folder. Rerunning the same protocol/repetition/load replaces its logs.
KEEP_BUILD_LOGS = True                 # False saves only stellar-core.log and client.log.
RESUME_SESSION = False                 # True: reuse VM source; check binaries once, not per protocol.
# Both modes run ALL configured trials into RESULTS_DIR; no automatic timestamp folder.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"          # Same custom image family.
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"  # Checkout/toolchain already in the image.
REMOTE_ROOT = "/home/tejas/stellar-gcp-load"
INSTANCE_PREFIX = "tsm-sc-load"         # Original fleet names; using e2-standard-4 VMs.
NUM_NODES = 10                         # Forget-IT requires 3f+1: 4, 7, 10, ...
REPLICA_ZONES = ["us-central1-c"] * NUM_NODES
CLIENT_ZONES = ["us-central1-c", "us-central1-c"]
MACHINE_TYPE = "e2-standard-4"
CLIENT_MACHINE_TYPE = "e2-standard-4"
NETWORK = "default"
BOOT_DISK_GB = 50                       # Existing image checkout and experiment logs.
CREATE_MISSING_VMS = True              # Reuse matching dedicated VMs if present.
DELETE_ALL_AFTER_FINAL_RUN = True     # True deletes all replica/client VMs in this run after collecting results.
USE_IAP = False                        # True if direct SSH is unavailable.
BUILD_JOBS = 2                         # make -j4 in the existing checkout on each replica VM.
MAX_PARALLEL_WORKERS = 48               # Parallel VM operations, as in the original runner.
STREAM_BUILD_OUTPUT = False            # Full compiler logs stay on disk; True also prints every line in Jupyter.
SOURCE_COMMIT = None                   # None pins local HEAD (must be pushed).

PROTOCOLS = ["tetrabft", "shabdiz"]
# simple_it = Opt-RBC, non-speculative. Optionally add "simple_it_bracha".
# SCP has a different native workload/ACK interface and is not comparable here.
WORKLOAD = "WORKLOAD_W"
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 80
MEASUREMENT_START_SEC = 30              # ACK completion window [30, CLIENT_DURATION_SEC).
MEASUREMENT_END_SEC = CLIENT_DURATION_SEC
STARTUP_TIMEOUT_SEC = 180
CLIENT_START_LEAD_SEC = 45              # Common wall-clock start across client VMs.
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Use >=3 for final paper measurements.
ITHS_BLOG_DELTA_MS = 1000               # Explicit: 2 Delta wait EACH sequential batch.
FORGET_IT_DELTA_MS = 1000               # Recovery timing; preserves current default.
SHABDIZ_FAST_PATH_ONLY = False          # True: memory-only block archive; keep stellar-core.log.
LOAD_POINTS = [
    # {"active_clients": 1, "client_threads": 1, "max_in_flight": 100},  # 100 total
    # {"active_clients": 1, "client_threads": 2, "max_in_flight": 75},   # 150 total
    # {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},  # 200 total
    # {"active_clients": 2, "client_threads": 2, "max_in_flight": 75},   # 300 total
    # {"active_clients": 2, "client_threads": 2, "max_in_flight": 100},  # 400 total
    # {"active_clients": 2, "client_threads": 2, "max_in_flight": 125},  # 500 total

    {"active_clients": 2, "client_threads": 2, "max_in_flight": 125},  # 500
    {"active_clients": 2, "client_threads": 2, "max_in_flight": 150},  # 600
    {"active_clients": 2, "client_threads": 2, "max_in_flight": 200},  # 800
    {"active_clients": 2, "client_threads": 2, "max_in_flight": 250},  # 1000

]
# By default VMs remain running for reuse. Deletion applies only to this run\'s fleet.
# ------------------------------------------------------

COMMIT_TAGS = {"shabdiz": ["[FAST COMMITTED]"],
               "iths": ["[IT-HS COMMITTED]", "[IT-HS-BLOG COMMITTED]"],
               "pbft": ["[PBFT COMMITTED-LOCAL]"], "tetrabft": ["[TETRA COMMITTED]"],
               "simple_it": ["[SIMPLE-IT COMMITTED]"],
               "simple_it_bracha": ["[SIMPLE-IT COMMITTED]"],
               "forget_it": ["[FORGET-IT COMMITTED]"]}


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600, stream_label=None):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        if stream_label is None or not STREAM_BUILD_OUTPUT:
            # Write directly to disk: no compiler-output accumulation in Jupyter or Python.
            subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
            return
        # Optional verbose debugging only; full compiler logs are always kept on disk.
        proc = subprocess.Popen(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, bufsize=1, start_new_session=True)
        expired = threading.Event()
        def expire():
            if proc.poll() is None:
                expired.set()
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        timer = threading.Timer(timeout, expire)
        timer.daemon = True
        timer.start()
        try:
            for line in proc.stdout:
                out.write(line)
                out.flush()
                print(f'[{stream_label}] {line.rstrip()}', flush=True)
            code = proc.wait()
            if expired.is_set():
                raise subprocess.TimeoutExpired(args, timeout)
            if code:
                raise subprocess.CalledProcessError(code, args)
        finally:
            timer.cancel()
            if proc.poll() is None:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            proc.wait()
            proc.stdout.close()


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    items = list(items)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_WORKERS, len(items))) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600, stream_label=None):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout, stream_label=stream_label)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def runtime_environment(protocol):
    # Same settings as the former header edits, now selected at process startup.
    return {
        'STELLAR_EXPERIMENT_PROTOCOL': protocol,
        'STELLAR_EXPERIMENT_NONPRUNE': 'false',
        'STELLAR_EXPERIMENT_WORKLOAD': WORKLOAD,
        'STELLAR_EXPERIMENT_BATCH_SIZE': str(SERVER_BATCH_SIZE),
        'STELLAR_EXPERIMENT_CLIENT_PORT': str(CLIENT_PORT),
        'STELLAR_EXPERIMENT_ITHS_DELTA_MS': str(ITHS_BLOG_DELTA_MS),
        'STELLAR_EXPERIMENT_FORGET_IT_DELTA_MS': str(FORGET_IT_DELTA_MS),
        'STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY': str(SHABDIZ_FAST_PATH_ONLY).lower(),
        'STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC': '2147483647',
    }


def runtime_command(args, protocol):
    return ['env', *(key + '=' + value for key, value in runtime_environment(protocol).items()), *args]


def require_runtime_config(header):
    if any(key not in header for key in runtime_environment(PROTOCOLS[0])):
        raise ValueError('Source lacks runtime experiment settings. Pull the updated source and '
                         'run once with RESUME_SESSION=False (and an updated SOURCE_COMMIT if set).')


def fleet(session):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    # Validate all existing VM identities before starting any cloud mutations.
    # Each worker then creates/starts and inspects one VM; ordering remains
    # node-01 ... node-N, client-01 ... so the ingress leader never changes.
    requested = []
    for role, zones in [("node", REPLICA_ZONES), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] not in ("TERMINATED", "RUNNING"):
                    raise RuntimeError(f"{name} is {old['status']}")
            elif not CREATE_MISSING_VMS:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            requested.append({"name": name, "zone": zone, "role": role, "index": index,
                              "machine_type": machine, "existing": matches[0] if matches else None})

    def provision_vm(spec):
        name, zone, machine = spec['name'], spec['zone'], spec['machine_type']
        old = spec['existing']
        if old is None:
            print(f'Creating {name} in {zone}...', flush=True)
            cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                  "--image", image["selfLink"], "--network", NETWORK,
                  "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                  "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-client-load", timeout=900)
        elif old['status'] == 'TERMINATED':
            cloud("instances", "start", name, "--zone", zone, timeout=900)
        vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
        if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
            raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
        disk_url = vm["disks"][0]["source"]
        disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                "--zone", zone, "--format=json"))
        if disk.get("sourceImage") != image["selfLink"]:
            raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
        if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
            cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
        print(f"{name}: {'created' if old is None else 'reused'}", flush=True)
        return {"name": name, "zone": zone, "role": spec['role'], "index": spec['index'],
                "ip": vm["networkInterfaces"][0]["networkIP"],
                "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]}

    vms = parallel(provision_vm, requested)
    # One coordinator writes the manifest after all workers finish.
    save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(NUM_NODES)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            raise RuntimeError(f"Firewall {rule} lacks required ports; use another INSTANCE_PREFIX")
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def launch_process(vm, root, args, lib, log, start_at=0):
    # Exec the binary so the recorded PID belongs to the actual process.
    command_line = shlex.join(args)
    log_path = log if log.startswith("/") else root + "/" + log
    lines = ['cd ' + shlex.quote(root),
             'export LD_LIBRARY_PATH=' + shlex.quote(lib) + ':"${LD_LIBRARY_PATH:-}"']
    if start_at:
        lines.append('python3 -c ' + shlex.quote(
            f'import time; time.sleep(max(0, {start_at!r} - time.time()))'))
    lines += ['date +%s.%N > started_at', 'exec ' + command_line]
    script = '\n'.join(lines) + '\n'
    code = f"from pathlib import Path; p=Path({root!r}); p.mkdir(parents=True, exist_ok=True); (p/'launch.sh').write_text({script!r})"
    ssh(vm, 'python3 -c ' + shlex.quote(code) + '\n' +
        'cd ' + shlex.quote(root) + '\nnohup bash ' + shlex.quote(root + '/launch.sh') +
        ' > ' + shlex.quote(log_path) + ' 2>&1 < /dev/null &\n' +
        'echo "$!" > ' + shlex.quote(root + '/pid') + '\ndisown')


def process_state(vm, root, *, check_ready=False):
    code = rf'''
import json, pathlib
root = pathlib.Path({root!r})
status = {{'done': False, 'ready': False, 'listener': False, 'returncode': None}}
pid = int((root/'pid').read_text())
try:
    process = pathlib.Path('/proc')/str(pid)
    alive = process.joinpath('stat').read_text().rsplit(')', 1)[1].split()[0] != 'Z'
    alive = alive and process.joinpath('cwd').resolve() == root.resolve()
except FileNotFoundError:
    alive = False
if not alive:
    status['done'] = True
    client_log = root/'client.log'
    if client_log.exists() and '=== Multi-thread Client Summary ===' in client_log.read_text(errors='replace'):
        status['returncode'] = 0
    else:
        status['error'] = 'Process exited before workload ended; inspect console/client log'
if (root/'started_at').exists():
    status['started_at'] = float((root/'started_at').read_text())
# Read-only readiness: authenticate the exact configured identities, not
# just a peer count or a log message. Works when replica logging is disabled.
if {check_ready!r} and alive:
    import re, urllib.request
    try:
        cfg = (root/'stellar-core.cfg').read_text()
        expected = set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^" ]+)"', cfg, re.M))
        port = int(re.search(r'^HTTP_PORT\s*=\s*(\d+)', cfg, re.M)[1])
        with urllib.request.urlopen(f'http://127.0.0.1:{{port}}/peers?fullkeys=true', timeout=5) as response:
            peers = json.load(response).get('authenticated_peers') or {{}}
        connected = {{p['id'] for direction in ('inbound', 'outbound')
                     for p in (peers.get(direction) or [])}}
        status['expected_peers'] = len(expected)
        status['connected_peers'] = len(expected & connected)
        status['missing_peers'] = sorted(expected - connected)
        status['ready'] = len(expected) == {NUM_NODES - 1} and expected <= connected
        # Confirm this process owns the client LISTEN socket without opening
        # a probe connection that would be mistaken for a workload client.
        sockets = set()
        for fd in process.joinpath('fd').iterdir():
            try:
                sockets.add(fd.readlink().as_posix())
            except FileNotFoundError:
                pass
        for table in ('tcp', 'tcp6'):
            net = process/'net'/table
            if net.exists():
                for row in net.read_text().splitlines()[1:]:
                    fields = row.split()
                    if (fields[3] == '0A' and int(fields[1].rsplit(':', 1)[1], 16) == {CLIENT_PORT}
                            and 'socket:[' + fields[9] + ']' in sockets):
                        status['listener'] = True
    except Exception as exc:
        status['probe_error'] = str(exc)
print(json.dumps(status))
'''
    return json.loads(python_remote(vm, code))

REMOTE_BUILD_HELPERS = r'''
import json, pathlib, subprocess, time
timings = {}
def timed(name, args, **kwargs):
    start = time.monotonic()
    print(name + ": starting", flush=True)
    try:
        subprocess.run(args, check=True, **kwargs)
    finally:
        timings[name] = round(time.monotonic() - start, 3)
        print(name + ": " + str(timings[name]) + " s", flush=True)
'''


def build_vm(vm, code, folder, saved_log=None):
    folder.mkdir(parents=True, exist_ok=True)
    log = folder / 'build.log'
    script = 'cd ' + shlex.quote(REMOTE_BUILD) + '\npython3 -c ' + shlex.quote(code)
    start = time.monotonic()
    try:
        ssh(vm, script, log=log, timeout=14400, stream_label=vm['name'])
    finally:
        if KEEP_BUILD_LOGS and saved_log is not None and log.is_file():
            saved_log.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(log, saved_log)
    result = {}
    with log.open(errors='replace') as output:
        for line in output:
            if line.startswith('BUILD_TIMINGS_JSON='):
                result = json.loads(line.partition('=')[2])
    result['wall_sec'] = round(time.monotonic() - start, 3)
    save_json(folder / 'build-timings.json', result)
    print(vm['name'] + ': ' + ('reused build' if result.get('reused') else
          ', '.join(f'{key}={value:.1f}s' for key, value in result.items()
                    if key.endswith('_sec'))), flush=True)


def build_replica(vm, header, folder):
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
# Restore old notebook-generated flags once; never rewrite for protocol/size changes.
if header.read_text() != canonical:
    header.write_text(canonical)
# Build once in the image's ORIGINAL checkout; unchanged inputs make this a no-op.
timed('make_sec', ['make', '-j{BUILD_JOBS}', '-o', 'Makefile', '-o', 'config.status',
                   'MAKE=make -o Makefile -o config.status'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-build.log'))


def build_client(vm, header, folder):
    # One workload-configurable client; compile only when source inputs changed.
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
if header.read_text() != canonical:
    header.write_text(canonical)
binary = pathlib.Path('shab_client')
inputs = [pathlib.Path(name) for name in ('shab_client.cpp', 'src/overlay/CustomYCSBWorkload.h',
          'src/overlay/CustomProtocolTypes.h', 'src/overlay/experiments/ExperimentConfig.h')]
if binary.is_file() and all(path.stat().st_mtime_ns <= binary.stat().st_mtime_ns for path in inputs):
    timings['reused'] = True
else:
    timed('client_compile_sec', ['g++', '-O2', '-std=c++17', '-pthread', '-Isrc',
                                 'shab_client.cpp', '-o', 'shab_client'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    # The common client is compiled once; retain its build log under the first protocol.
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-client-build.log'))


def make_configs(builder, art, trial, node_ips, folder, protocol):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {NUM_NODES}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({NUM_NODES}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{NUM_NODES})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
        shlex.join(runtime_command(["python3", "-c", code], protocol)),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def collect_trial_logs(vm, root, local, config):
    filenames = (['stellar-core.log']
                 if vm['role'] == 'node' else ['client.log'])
    diagnostics = []
    # One compressed transfer per VM; omit databases/buckets and retain partial logs on failure.
    code = f'''
import io, json, pathlib, tarfile
root = pathlib.Path({root!r})
root.mkdir(parents=True, exist_ok=True)
missing = []
with tarfile.open(root/'collected-logs.tgz', 'w:gz', compresslevel=1) as archive:
    for name in {filenames!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
        else: missing.append(name)
    for name in {diagnostics!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
    data = json.dumps({{'missing': missing}}).encode()
    member = tarfile.TarInfo('collection.json'); member.size = len(data)
    archive.addfile(member, io.BytesIO(data))
'''
    python_remote(vm, code)
    local.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='stellar-log-transfer-') as transfer_dir:
        archive_path = Path(transfer_dir) / 'collected-logs.tgz'
        scp(vm, root + '/collected-logs.tgz', archive_path, download=True)
        with tarfile.open(archive_path, 'r:gz') as archive:
            missing = json.load(archive.extractfile('collection.json'))['missing']
            for name in filenames + diagnostics:
                if name in archive.getnames():
                    member = archive.getmember(name)
                    if not member.isfile():
                        raise RuntimeError('Expected a regular log file: ' + name)
                    with archive.extractfile(member) as source, (local / name).open('wb') as output:
                        if name == 'client.log':
                            # Keep plot metadata in the log itself, without extra JSON files.
                            output.write(('# RUN_CONFIG_JSON=' + json.dumps(config, allow_nan=False) + '\n').encode())
                        shutil.copyfileobj(source, output)
    if missing:
        raise RuntimeError(vm['name'] + ': missing ' + ', '.join(missing))


def run_trial(vms, builder, art, protocol, load, repetition, load_index, session, common, client_art=None):
    label = trial_label(protocol, load, repetition, load_index)
    folder = session / label
    folder.mkdir(parents=True)
    logs_folder = Path(RESULTS_DIR).expanduser().resolve() / label
    logs_folder.mkdir(parents=True, exist_ok=True)
    # A rerun replaces this exact trial. Do not leave old client data behind if
    # the new run fails before collecting logs, and never touch other trials.
    for pattern in ('node-*/stellar-core.log', 'client-*/client.log'):
        for old_log in logs_folder.glob(pattern):
            old_log.unlink()
    # Each retry gets a new directory; preserve the failed attempt logs.
    trial = REMOTE_SESSION + "/runs/" + label + "/attempt-" + uuid.uuid4().hex
    client_art = client_art or art
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load['active_clients']]
    config = {**common, **load, "protocol": protocol, "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    states = []
    cleanup_errors = []
    timings = {}
    setup_start = time.monotonic()
    print(f'Starting {label}: setup, then {CLIENT_START_LEAD_SEC}s scheduled client lead.', flush=True)
    try:
        parallel(stop_vm_processes, vms)
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder, protocol)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join(runtime_command([art + "/stellar-core", "new-db", "--conf",
                                            root + "/stellar-core.cfg"], protocol)),
                log=folder / f"node-{vm['index']}-init.log")
            launch_process(vm, root, runtime_command(
                [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"], protocol),
                           art + "/lib",
                           "console.log")
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        last_connectivity = None
        while True:
            status = parallel(lambda v: process_state(v, trial + f"/nodes/node{v['index']}", check_ready=True), nodes)
            if any(s['done'] or s.get('error') for s in status):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise RuntimeError('Replica exited during startup: ' + str(result['replica_outcomes']))
            connectivity = (sum(s['ready'] for s in status),
                            min(s.get('connected_peers', 0) for s in status))
            if connectivity != last_connectivity:
                print(f"Connectivity: {connectivity[0]}/{len(nodes)} replicas fully connected; "
                      f"minimum {connectivity[1]}/{len(nodes)-1} authenticated peers", flush=True)
                last_connectivity = connectivity
            if all(s['ready'] for s in status) and status[0]['listener']:
                break
            if time.monotonic() >= deadline:
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise TimeoutError("Full peer connectivity/client listener timed out: " +
                                   str(result['replica_outcomes']))
            time.sleep(3)
        start_at = time.time() + CLIENT_START_LEAD_SEC
        timings['setup_sec'] = round(time.monotonic() - setup_start, 3)
        config['scheduled_client_start'] = start_at
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            launch_process(vm, trial + f"/client-{vm['index']}",
                runtime_command([client_art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                 "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                 str(SEND_INTERVAL_US), str(load['client_threads'])], protocol),
                client_art + "/lib", "client.log", start_at=start_at)
        parallel(start_client, clients)
        if time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        workload_end = start_at + CLIENT_DURATION_SEC
        while time.time() < workload_end:
            # Poll replicas and clients in one batch; neither group depends on the other.
            observed = parallel(lambda pair: process_state(*pair), owned)
            replica_states, states = observed[:len(nodes)], observed[len(nodes):]
            if any(s['done'] or s.get('error') for s in replica_states):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], replica_states))
                raise RuntimeError('Replica exited during workload: ' + str(result['replica_outcomes']))
            if any(s.get('error') for s in states):
                raise RuntimeError('Client process failed: ' + str(states))
            if all(s['done'] for s in states):
                break
            time.sleep(min(3, max(0, workload_end - time.time())))
        result['client_outcomes'] = states
        if any('started_at' not in s for s in states):
            raise RuntimeError('A client never started: ' + str(states))
        starts = [s['started_at'] for s in states]
        result['client_start_skew_sec'] = max(starts) - min(starts)
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or (s['done'] and s['returncode'] != 0) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'ok'
        result['stop_reason'] = 'workload_duration_reached' if time.time() >= workload_end else 'clients_completed'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        stop_start = time.monotonic()
        def stop(pair):
            vm, root = pair
            try:
                stop_vm_processes(vm)
            except Exception as exc:
                cleanup_errors.append(f"{vm['name']}: {exc}")
        parallel(stop, owned)
        timings['stop_sec'] = round(time.monotonic() - stop_start, 3)
        collect_start = time.monotonic()
        def collect(pair):
            vm, root = pair
            local = logs_folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            collect_trial_logs(vm, root, local, config)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        timings['collect_sec'] = round(time.monotonic() - collect_start, 3)
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        if result['status'] in ('ok', 'drain_timeout'):
            def count_commits(vm):
                with (logs_folder / f"node-{vm['index']}" / 'stellar-core.log').open(errors='replace') as log:
                    return sum(line.count(tag) for line in log for tag in COMMIT_TAGS[protocol])
            counts = parallel(count_commits, nodes)
            result['replica_commits'] = counts
            if not all(counts):
                result.update(status='failed', error='A replica logged no commits')
        result['timings_sec'] = timings
        save_json(folder / "result.json", result)
        print(f"Cleanup {label}: stop={timings['stop_sec']:.1f}s, logs={timings['collect_sec']:.1f}s", flush=True)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError(result.get('error', 'Trial cleanup failed') + '\nCleanup: ' +
                           '; '.join(cleanup_errors) + '\nLogs: ' + str(logs_folder))
    return result


def trial_label(protocol, load, repetition, index):
    return (f"{protocol}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def stop_vm_processes(vm, *, include_builds=False):
    # These are dedicated experiment VMs. Restrict cleanup to the SSH user's processes.
    # Bracketed dots keep pkill from matching the cleanup command itself.
    pattern = re.escape(REMOTE_ROOT) + r'/[^ ]*/(launch[.]sh|worker[.]py supervise )'
    script = """
kill_matching() {
    pkill -u "$(id -u)" "$@" || { rc=$?; [ "$rc" -eq 1 ]; }
}
kill_matching -TERM -f PATTERN
kill_matching -TERM -x stellar-core
kill_matching -TERM -x shab_client
if pgrep -x stellar-core >/dev/null || pgrep -x shab_client >/dev/null; then
    sleep 1
fi
kill_matching -KILL -x stellar-core
kill_matching -KILL -x shab_client
kill_matching -KILL -f PATTERN
""".replace('PATTERN', shlex.quote(pattern))
    if include_builds:
        # Kill old notebook build supervisors first, so they cannot start another
        # make/compiler after cleanup. The bracket prevents matching this script.
        # Dedicated VMs may have orphaned children after a kernel/SSH interruption.
        build_cleanup = """
kill_matching -KILL -f '[B]UILD_TIMINGS_JSON='
kill_matching -KILL -x '(make|gmake|ninja|ninja-build|cmake|cargo)(-[0-9.]+)?'
kill_matching -KILL -x '(gcc|g[+][+]|cc|c[+][+]|clang|clang[+][+]|cc1|cc1plus|collect2|lto-wrapper|lto1|as|ld|ld[.]bfd|ld[.]gold|ld[.]lld|lld|rustc|rustdoc|build-script-bu)(-[0-9.]+)?'
"""
        # Insert after the shell helper, before stopping replicas and clients.
        script = script.replace("\n}\n", "\n}\n" + build_cleanup, 1)
    ssh(vm, script, timeout=30)


def prepare_build(vm, sha, folder):
    # Update the existing checkout once. Never copy it, clean it, or reinstall dependencies.
    script = f"""
cd {shlex.quote(IMAGE_REPO)}
test -d .git
test -f Makefile
if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
    export GIT_TERMINAL_PROMPT=0
    # The sweep itself edits this header; clear those generated flags before pulling.
    if ! git diff --quiet HEAD -- src/overlay/experiments/ExperimentConfig.h; then
        git restore --source=HEAD -- src/overlay/experiments/ExperimentConfig.h
    fi
    git pull --ff-only origin {shlex.quote(sha)}
    if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
        echo "Existing checkout is ahead of the session commit; refusing to mix source versions." >&2
        exit 1
    fi
fi
printf 'Using existing configured checkout: %s\\n' "$PWD"
"""
    ssh(vm, script, log=folder / 'prepare-build.log', timeout=600)


def finish_session(session):
    print('Finished:', Path(RESULTS_DIR).expanduser().resolve(),
          '\nRun the GCP client-load plotting cell in PostProcess.ipynb with the same RESULTS_DIR.')
    saved = json.loads((session / 'fleet.json').read_text()) if (session / 'fleet.json').exists() else {'vms': []}
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [v for v in saved['vms'] if (v['name'], v['zone']) in existing]
        print('Deleting remaining replica/client VMs:', ', '.join(v['name'] for v in remaining), flush=True)
        parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
                 remaining)
    else:
        print('VMs kept running for reuse (DELETE_ALL_AFTER_FINAL_RUN=False).')


def main_gcp_load(work_dir):
    global REMOTE_SESSION, REMOTE_BUILD
    repo = REPO.expanduser().resolve()
    if type(RESUME_SESSION) is not bool:
        raise ValueError('RESUME_SESSION must be True or False')
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if NUM_NODES < 4 or len(REPLICA_ZONES) != NUM_NODES:
        raise ValueError("Specify one zone per replica and at least four replicas")
    if 'forget_it' in PROTOCOLS and (NUM_NODES - 1) % 3:
        raise ValueError("Forget-IT requires n=3f+1 (4, 7, 10, ...); 8 is unsupported")
    if not PROTOCOLS or len(set(PROTOCOLS)) != len(PROTOCOLS) or set(PROTOCOLS) - COMMIT_TAGS.keys():
        raise ValueError("Choose distinct custom-client protocols from COMMIT_TAGS; SCP is diagnostic only")
    if not (0 <= MEASUREMENT_START_SEC < MEASUREMENT_END_SEC <= CLIENT_DURATION_SEC):
        raise ValueError("Invalid measurement window")
    if type(MAX_PARALLEL_WORKERS) is not int or MAX_PARALLEL_WORKERS < 1:
        raise ValueError('MAX_PARALLEL_WORKERS must be a positive integer')
    if min(REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE, ITHS_BLOG_DELTA_MS, FORGET_IT_DELTA_MS) <= 0:
        raise ValueError("Counts, batch size, and Delta must be positive")
    if not re.fullmatch(r'WORKLOAD_(W|R0)', WORKLOAD):
        raise ValueError("This comparison uses write-only transactions; choose WORKLOAD_W or WORKLOAD_R0")
    if not LOAD_POINTS:
        raise ValueError("No client load points")
    for load in LOAD_POINTS:
        if set(load) != {'active_clients','client_threads','max_in_flight'} or min(load.values()) <= 0:
            raise ValueError(f"Invalid load point: {load}")
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError("Insufficient client VMs or aggregate in-flight capacity for one batch")
    sha = header = None
    if not RESUME_SESSION:
        sha = command(['git', '-C', repo, 'rev-parse', (SOURCE_COMMIT or 'HEAD') + '^{commit}']).strip()
        header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
        require_runtime_config(header)
    session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    # Only remote working paths need unique IDs. Local results use the fixed input.
    session = Path(work_dir)
    REMOTE_SESSION = REMOTE_ROOT + '/' + session_id
    REMOTE_BUILD = IMAGE_REPO
    common = {'schema_version': 1, 'session': session_id, 'source_commit': sha, 'num_nodes': NUM_NODES,
              'workload': WORKLOAD, 'batch_size': SERVER_BATCH_SIZE, 'client_duration_sec': CLIENT_DURATION_SEC,
              'measurement_start_sec': MEASUREMENT_START_SEC, 'measurement_end_sec': MEASUREMENT_END_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'iths_blog_delta_ms': ITHS_BLOG_DELTA_MS, 'forget_it_delta_ms': FORGET_IT_DELTA_MS,
              'shabdiz_fast_path_only': SHABDIZ_FAST_PATH_ONLY, 'forced_collection': False,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'replica_zones': REPLICA_ZONES, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': 0}
    manifest = {**common, 'protocols': PROTOCOLS, 'load_points': LOAD_POINTS,
                'repetitions': REPETITIONS, 'instance_prefix': INSTANCE_PREFIX,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN, 'resume_session': RESUME_SESSION}
    session.mkdir(parents=True, exist_ok=True)
    save_json(session / 'session.json', manifest)
    print('Results:', Path(RESULTS_DIR).expanduser().resolve(), flush=True)
    trials = [(protocol, repetition, index, load) for protocol in PROTOCOLS
              for repetition in range(1, REPETITIONS+1) for index, load in enumerate(LOAD_POINTS, 1)]
    results = []
    save_json(session / 'results.json', results)
    print(f"{len(trials)} trials; "
          f"{len(trials)*CLIENT_DURATION_SEC/3600:.1f} "
          'hours of workload, plus builds and setup; pkill at workload end.', flush=True)
    print(f'IT-HS Delta={ITHS_BLOG_DELTA_MS} ms: its per-batch proposal wait is {2*ITHS_BLOG_DELTA_MS} ms.')
    vms = fleet(session)
    parallel(wait_ssh, vms)
    common['replica_zones'] = REPLICA_ZONES
    common['client_zones'] = CLIENT_ZONES
    common['resolved_image'] = json.loads((session/'fleet.json').read_text())['image']
    try:
        print('Stopping stale replicas, clients, and builds on all experiment VMs...', flush=True)
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
        def initialize_vm(vm):
            sync = ssh(vm, shlex.join(['mkdir', '-p', REMOTE_SESSION]) + '\n' +
                       'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before running IT-HS/scheduled clients")
        parallel(initialize_vm, vms)
        builder = next(v for v in vms if v['role']=='node')
        client_art = IMAGE_REPO
        def prepare_vm(vm):
            folder = session / 'preparation' / vm['name']
            folder.mkdir(parents=True, exist_ok=True)
            prepare_build(vm, sha, folder)
        if RESUME_SESSION:
            print('Reusing VM source; checking common replica/client builds once.', flush=True)
            def existing_source(vm):
                check = 'test -f shab_client.cpp' if vm['role'] == 'client' else 'test -f Makefile'
                return ssh(vm, 'cd ' + shlex.quote(IMAGE_REPO) + '\n' + check + '\ngit rev-parse HEAD').strip()
            sources = parallel(existing_source, vms)
            if len(set(sources)) != 1:
                raise RuntimeError('VM source revisions differ. Run once with RESUME_SESSION=False to update them.')
            sha = sources[0]
            header = ssh(builder, 'cd ' + shlex.quote(IMAGE_REPO) +
                         '\ngit show HEAD:src/overlay/experiments/ExperimentConfig.h')
            require_runtime_config(header)
        else:
            print(f'Updating existing {IMAGE_REPO} once before the common build.', flush=True)
            parallel(prepare_vm, vms)
        common['source_commit'] = manifest['source_commit'] = sha
        save_json(session / 'session.json', manifest)
        print('Source:', sha, flush=True)
        # A single binary handles all protocols and node counts. make is incremental;
        # the client timestamp check also runs in resume mode to avoid stale binaries.
        print(f'Checking common builds once ({BUILD_JOBS} make jobs per replica)...', flush=True)
        build_folder = session / 'builds' / PROTOCOLS[0]
        art = IMAGE_REPO + '/src'
        build_start = time.monotonic()
        def build_common(vm):
            folder = build_folder / vm['name']
            if vm['role'] == 'node':
                build_replica(vm, header, folder)
            else:
                build_client(vm, header, folder)
        parallel(build_common, vms)
        print(f'Common binaries ready in {time.monotonic() - build_start:.1f}s', flush=True)
        for protocol in PROTOCOLS:
            protocol_trials = [trial for trial in trials if trial[0] == protocol]
            print(f'{protocol}: reusing common binaries; selecting runtime settings.', flush=True)
            for _, repetition, index, load in protocol_trials:
                result = run_trial(vms, builder, art, protocol, load, repetition, index, session, common, client_art)
                results.append(result)
                save_json(session / 'results.json', results)
    finally:
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
    failed = [result for result in results if result['status'] not in ('ok', 'drain_timeout')]
    if failed:
        print(f'{len(failed)} trials failed; VMs kept. Set RESUME_SESSION=True to rerun all trials with existing builds.')
    else:
        finish_session(session)
    return Path(RESULTS_DIR).expanduser().resolve()


def run_gcp_load():
    Path(RESULTS_DIR).expanduser().resolve().mkdir(parents=True, exist_ok=True)
    # Config archives, setup logs, timing files, and status JSON are temporary.
    # The fixed output contains only per-trial replica/client logs and optional builds.
    with tempfile.TemporaryDirectory(prefix='stellar-gcp-load-') as work_dir:
        return main_gcp_load(work_dir)


GCP_LOAD_SESSION = run_gcp_load()



Results: /home/tejas/work/experiments/gcp-client-load/ClientThputLatency
8 trials; 0.2 hours of workload, plus builds and setup; pkill at workload end.
IT-HS Delta=1000 ms: its per-batch proposal wait is 2000 ms.
Creating tsm-sc-load-node-01 in us-central1-c...Creating tsm-sc-load-node-02 in us-central1-c...

Creating tsm-sc-load-node-03 in us-central1-c...
Creating tsm-sc-load-node-04 in us-central1-c...
Creating tsm-sc-load-node-05 in us-central1-c...
Creating tsm-sc-load-node-06 in us-central1-c...
Creating tsm-sc-load-node-07 in us-central1-c...
Creating tsm-sc-load-node-08 in us-central1-c...
Creating tsm-sc-load-node-09 in us-central1-c...
Creating tsm-sc-load-node-10 in us-central1-c...
Creating tsm-sc-load-client-01 in us-central1-c...
Creating tsm-sc-load-client-02 in us-central1-c...
tsm-sc-load-node-02: created
tsm-sc-load-node-10: created
tsm-sc-load-node-07: created
tsm-sc-load-node-06: created
tsm-sc-load-node-05: created
tsm-sc-load-node-04: created
tsm-sc-load-node-01: 

# Main num nodes experiment

In [ ]:
"""GCP throughput and ACK latency versus replica count.
Run this cell independently; it does not run or depend on the client-load cell.
Create the largest fleet in parallel, build once, run every protocol at each
size, then delete surplus replicas before moving to the next smaller size.
Only this cell's replica/client names are managed. No automatic git commit/push.
"""
import concurrent.futures
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import socket
import subprocess
import tarfile
import tempfile
import threading
import time
import uuid

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Updated local stellar-core checkout; HEAD must be pushed.
RESULTS_DIR = Path.home() / "work/experiments/gcp-num-nodes/SystemSize"
# Fixed directory: n49/shabdiz/repeat1/load1_.../{node-*,client-*}/*.log.
KEEP_BUILD_LOGS = True
RESUME_SESSION = False                 # True reuses VM source, but reruns every configured trial.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"
REMOTE_ROOT = "/home/tejas/stellar-gcp-size"
INSTANCE_PREFIX = "tsm-sc-size"         # Separate from the e2-standard-4 client-load fleet.
# Nearest 3f+1 sizes to 48, 32, 16, 8, 4, so Forget-IT runs at EVERY size too.
NUM_NODES_LIST = [49, 31, 16, 7, 4]      # Strictly descending; replica indices/zones stay fixed.
# One zone for each replica in the LARGEST fleet. Smaller sizes use its prefix.
REPLICA_ZONES = ["us-central1-c"] * max(NUM_NODES_LIST)
CLIENT_ZONES = ["us-central1-c", "us-central1-c"]  # Additional VMs, retained across all sizes.
MACHINE_TYPE = "e2-standard-2"
CLIENT_MACHINE_TYPE = "e2-standard-2"
NETWORK = "default"
BOOT_DISK_GB = 50
CREATE_MISSING_VMS = True
DELETE_ALL_AFTER_FINAL_RUN = True       # After successful n=4 runs, delete the remaining 4 + 2 VMs.
# Surplus replicas are always deleted after all protocols at a size finish successfully.
USE_IAP = False
BUILD_JOBS = 2                         # Incremental make -j2 once per replica; no autogen/configure.
MAX_PARALLEL_WORKERS = 48
STREAM_BUILD_OUTPUT = False
SOURCE_COMMIT = None                   # None pins the local HEAD once for the whole sweep.

PROTOCOLS = ["shabdiz", "tetrabft", "simple_it", "forget_it"]
# Selected throughput/latency operating points from the client-load experiment.
# These remain fixed per protocol across sizes, rather than claiming peak throughput at every size.
PROTOCOL_LOADS = {
    "shabdiz":   {"active_clients": 2, "client_threads": 2, "max_in_flight": 100},  # 400
    "tetrabft":  {"active_clients": 2, "client_threads": 2, "max_in_flight": 150},  # 600
    "simple_it": {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},  # 200
    "forget_it": {"active_clients": 2, "client_threads": 2, "max_in_flight": 75},   # 300
}
# To add another custom-client protocol, supply its measured load above too.
WORKLOAD = "WORKLOAD_W"
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 80
MEASUREMENT_START_SEC = 30
MEASUREMENT_END_SEC = CLIENT_DURATION_SEC
STARTUP_TIMEOUT_SEC = 180
CLIENT_START_LEAD_SEC = 45
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Set >=3 for repeatability/error bars.
ITHS_BLOG_DELTA_MS = 1000
FORGET_IT_DELTA_MS = 1000
SHABDIZ_FAST_PATH_ONLY = False          # True: memory-only block archive; keep stellar-core.log.
# ------------------------------------------------------

COMMIT_TAGS = {"shabdiz": ["[FAST COMMITTED]"],
               "iths": ["[IT-HS COMMITTED]", "[IT-HS-BLOG COMMITTED]"],
               "pbft": ["[PBFT COMMITTED-LOCAL]"], "tetrabft": ["[TETRA COMMITTED]"],
               "simple_it": ["[SIMPLE-IT COMMITTED]"],
               "simple_it_bracha": ["[SIMPLE-IT COMMITTED]"],
               "forget_it": ["[FORGET-IT COMMITTED]"]}


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600, stream_label=None):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        if stream_label is None or not STREAM_BUILD_OUTPUT:
            # Write directly to disk: no compiler-output accumulation in Jupyter or Python.
            subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
            return
        # Optional verbose debugging only; full compiler logs are always kept on disk.
        proc = subprocess.Popen(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, bufsize=1, start_new_session=True)
        expired = threading.Event()
        def expire():
            if proc.poll() is None:
                expired.set()
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        timer = threading.Timer(timeout, expire)
        timer.daemon = True
        timer.start()
        try:
            for line in proc.stdout:
                out.write(line)
                out.flush()
                print(f'[{stream_label}] {line.rstrip()}', flush=True)
            code = proc.wait()
            if expired.is_set():
                raise subprocess.TimeoutExpired(args, timeout)
            if code:
                raise subprocess.CalledProcessError(code, args)
        finally:
            timer.cancel()
            if proc.poll() is None:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            proc.wait()
            proc.stdout.close()


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    items = list(items)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_WORKERS, len(items))) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600, stream_label=None):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout, stream_label=stream_label)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def runtime_environment(protocol):
    # Same settings as the former header edits, now selected at process startup.
    return {
        'STELLAR_EXPERIMENT_PROTOCOL': protocol,
        'STELLAR_EXPERIMENT_NONPRUNE': 'false',
        'STELLAR_EXPERIMENT_WORKLOAD': WORKLOAD,
        'STELLAR_EXPERIMENT_BATCH_SIZE': str(SERVER_BATCH_SIZE),
        'STELLAR_EXPERIMENT_CLIENT_PORT': str(CLIENT_PORT),
        'STELLAR_EXPERIMENT_ITHS_DELTA_MS': str(ITHS_BLOG_DELTA_MS),
        'STELLAR_EXPERIMENT_FORGET_IT_DELTA_MS': str(FORGET_IT_DELTA_MS),
        'STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY': str(SHABDIZ_FAST_PATH_ONLY).lower(),
        'STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC': '2147483647',
    }


def runtime_command(args, protocol):
    return ['env', *(key + '=' + value for key, value in runtime_environment(protocol).items()), *args]


def require_runtime_config(header):
    if any(key not in header for key in runtime_environment(PROTOCOLS[0])):
        raise ValueError('Source lacks runtime experiment settings. Pull the updated source and '
                         'run once with RESUME_SESSION=False (and an updated SOURCE_COMMIT if set).')


def fleet(session):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    # Validate all existing VM identities before starting any cloud mutations.
    # Each worker then creates/starts and inspects one VM; ordering remains
    # node-01 ... node-N, client-01 ... so the ingress leader never changes.
    requested = []
    for role, zones in [("node", REPLICA_ZONES), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] not in ("TERMINATED", "RUNNING"):
                    raise RuntimeError(f"{name} is {old['status']}")
            elif not CREATE_MISSING_VMS:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            requested.append({"name": name, "zone": zone, "role": role, "index": index,
                              "machine_type": machine, "existing": matches[0] if matches else None})

    def provision_vm(spec):
        name, zone, machine = spec['name'], spec['zone'], spec['machine_type']
        old = spec['existing']
        if old is None:
            print(f'Creating {name} in {zone}...', flush=True)
            cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                  "--image", image["selfLink"], "--network", NETWORK,
                  "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                  "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-client-load", timeout=900)
        elif old['status'] == 'TERMINATED':
            cloud("instances", "start", name, "--zone", zone, timeout=900)
        vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
        if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
            raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
        disk_url = vm["disks"][0]["source"]
        disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                "--zone", zone, "--format=json"))
        if disk.get("sourceImage") != image["selfLink"]:
            raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
        if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
            cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
        print(f"{name}: {'created' if old is None else 'reused'}", flush=True)
        return {"name": name, "zone": zone, "role": spec['role'], "index": spec['index'],
                "ip": vm["networkInterfaces"][0]["networkIP"],
                "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]}

    vms = parallel(provision_vm, requested)
    # One coordinator writes the manifest after all workers finish.
    save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(NUM_NODES)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            raise RuntimeError(f"Firewall {rule} lacks required ports; use another INSTANCE_PREFIX")
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def launch_process(vm, root, args, lib, log, start_at=0):
    # Exec the binary so the recorded PID belongs to the actual process.
    command_line = shlex.join(args)
    log_path = log if log.startswith("/") else root + "/" + log
    lines = ['cd ' + shlex.quote(root),
             'export LD_LIBRARY_PATH=' + shlex.quote(lib) + ':"${LD_LIBRARY_PATH:-}"']
    if start_at:
        lines.append('python3 -c ' + shlex.quote(
            f'import time; time.sleep(max(0, {start_at!r} - time.time()))'))
    lines += ['date +%s.%N > started_at', 'exec ' + command_line]
    script = '\n'.join(lines) + '\n'
    code = f"from pathlib import Path; p=Path({root!r}); p.mkdir(parents=True, exist_ok=True); (p/'launch.sh').write_text({script!r})"
    ssh(vm, 'python3 -c ' + shlex.quote(code) + '\n' +
        'cd ' + shlex.quote(root) + '\nnohup bash ' + shlex.quote(root + '/launch.sh') +
        ' > ' + shlex.quote(log_path) + ' 2>&1 < /dev/null &\n' +
        'echo "$!" > ' + shlex.quote(root + '/pid') + '\ndisown')


def process_state(vm, root, *, check_ready=False):
    code = rf'''
import json, pathlib
root = pathlib.Path({root!r})
status = {{'done': False, 'ready': False, 'listener': False, 'returncode': None}}
pid = int((root/'pid').read_text())
try:
    process = pathlib.Path('/proc')/str(pid)
    alive = process.joinpath('stat').read_text().rsplit(')', 1)[1].split()[0] != 'Z'
    alive = alive and process.joinpath('cwd').resolve() == root.resolve()
except FileNotFoundError:
    alive = False
if not alive:
    status['done'] = True
    client_log = root/'client.log'
    if client_log.exists() and '=== Multi-thread Client Summary ===' in client_log.read_text(errors='replace'):
        status['returncode'] = 0
    else:
        status['error'] = 'Process exited before workload ended; inspect console/client log'
if (root/'started_at').exists():
    status['started_at'] = float((root/'started_at').read_text())
# Read-only readiness: authenticate the exact configured identities, not
# just a peer count or a log message. Works when replica logging is disabled.
if {check_ready!r} and alive:
    import re, urllib.request
    try:
        cfg = (root/'stellar-core.cfg').read_text()
        expected = set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^" ]+)"', cfg, re.M))
        port = int(re.search(r'^HTTP_PORT\s*=\s*(\d+)', cfg, re.M)[1])
        with urllib.request.urlopen(f'http://127.0.0.1:{{port}}/peers?fullkeys=true', timeout=5) as response:
            peers = json.load(response).get('authenticated_peers') or {{}}
        connected = {{p['id'] for direction in ('inbound', 'outbound')
                     for p in (peers.get(direction) or [])}}
        status['expected_peers'] = len(expected)
        status['connected_peers'] = len(expected & connected)
        status['missing_peers'] = sorted(expected - connected)
        status['ready'] = len(expected) == {NUM_NODES - 1} and expected <= connected
        # Confirm this process owns the client LISTEN socket without opening
        # a probe connection that would be mistaken for a workload client.
        sockets = set()
        for fd in process.joinpath('fd').iterdir():
            try:
                sockets.add(fd.readlink().as_posix())
            except FileNotFoundError:
                pass
        for table in ('tcp', 'tcp6'):
            net = process/'net'/table
            if net.exists():
                for row in net.read_text().splitlines()[1:]:
                    fields = row.split()
                    if (fields[3] == '0A' and int(fields[1].rsplit(':', 1)[1], 16) == {CLIENT_PORT}
                            and 'socket:[' + fields[9] + ']' in sockets):
                        status['listener'] = True
    except Exception as exc:
        status['probe_error'] = str(exc)
print(json.dumps(status))
'''
    return json.loads(python_remote(vm, code))

REMOTE_BUILD_HELPERS = r'''
import json, pathlib, subprocess, time
timings = {}
def timed(name, args, **kwargs):
    start = time.monotonic()
    print(name + ": starting", flush=True)
    try:
        subprocess.run(args, check=True, **kwargs)
    finally:
        timings[name] = round(time.monotonic() - start, 3)
        print(name + ": " + str(timings[name]) + " s", flush=True)
'''


def build_vm(vm, code, folder, saved_log=None):
    folder.mkdir(parents=True, exist_ok=True)
    log = folder / 'build.log'
    script = 'cd ' + shlex.quote(REMOTE_BUILD) + '\npython3 -c ' + shlex.quote(code)
    start = time.monotonic()
    try:
        ssh(vm, script, log=log, timeout=14400, stream_label=vm['name'])
    finally:
        if KEEP_BUILD_LOGS and saved_log is not None and log.is_file():
            saved_log.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(log, saved_log)
    result = {}
    with log.open(errors='replace') as output:
        for line in output:
            if line.startswith('BUILD_TIMINGS_JSON='):
                result = json.loads(line.partition('=')[2])
    result['wall_sec'] = round(time.monotonic() - start, 3)
    save_json(folder / 'build-timings.json', result)
    print(vm['name'] + ': ' + ('reused build' if result.get('reused') else
          ', '.join(f'{key}={value:.1f}s' for key, value in result.items()
                    if key.endswith('_sec'))), flush=True)


def build_replica(vm, header, folder):
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
# Restore old notebook-generated flags once; never rewrite for protocol/size changes.
if header.read_text() != canonical:
    header.write_text(canonical)
# Build once in the image's ORIGINAL checkout; unchanged inputs make this a no-op.
timed('make_sec', ['make', '-j{BUILD_JOBS}', '-o', 'Makefile', '-o', 'config.status',
                   'MAKE=make -o Makefile -o config.status'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-build.log'))


def build_client(vm, header, folder):
    # One workload-configurable client; compile only when source inputs changed.
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
if header.read_text() != canonical:
    header.write_text(canonical)
binary = pathlib.Path('shab_client')
inputs = [pathlib.Path(name) for name in ('shab_client.cpp', 'src/overlay/CustomYCSBWorkload.h',
          'src/overlay/CustomProtocolTypes.h', 'src/overlay/experiments/ExperimentConfig.h')]
if binary.is_file() and all(path.stat().st_mtime_ns <= binary.stat().st_mtime_ns for path in inputs):
    timings['reused'] = True
else:
    timed('client_compile_sec', ['g++', '-O2', '-std=c++17', '-pthread', '-Isrc',
                                 'shab_client.cpp', '-o', 'shab_client'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    # The common client is compiled once; retain its build log under the first protocol.
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-client-build.log'))


def make_configs(builder, art, trial, node_ips, folder, protocol):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {NUM_NODES}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({NUM_NODES}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{NUM_NODES})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
        shlex.join(runtime_command(["python3", "-c", code], protocol)),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def collect_trial_logs(vm, root, local, config):
    filenames = (['stellar-core.log']
                 if vm['role'] == 'node' else ['client.log'])
    diagnostics = []
    # One compressed transfer per VM; omit databases/buckets and retain partial logs on failure.
    code = f'''
import io, json, pathlib, tarfile
root = pathlib.Path({root!r})
root.mkdir(parents=True, exist_ok=True)
missing = []
with tarfile.open(root/'collected-logs.tgz', 'w:gz', compresslevel=1) as archive:
    for name in {filenames!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
        else: missing.append(name)
    for name in {diagnostics!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
    data = json.dumps({{'missing': missing}}).encode()
    member = tarfile.TarInfo('collection.json'); member.size = len(data)
    archive.addfile(member, io.BytesIO(data))
'''
    python_remote(vm, code)
    local.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='stellar-log-transfer-') as transfer_dir:
        archive_path = Path(transfer_dir) / 'collected-logs.tgz'
        scp(vm, root + '/collected-logs.tgz', archive_path, download=True)
        with tarfile.open(archive_path, 'r:gz') as archive:
            missing = json.load(archive.extractfile('collection.json'))['missing']
            for name in filenames + diagnostics:
                if name in archive.getnames():
                    member = archive.getmember(name)
                    if not member.isfile():
                        raise RuntimeError('Expected a regular log file: ' + name)
                    with archive.extractfile(member) as source, (local / name).open('wb') as output:
                        if name == 'client.log':
                            # Keep plot metadata in the log itself, without extra JSON files.
                            output.write(('# RUN_CONFIG_JSON=' + json.dumps(config, allow_nan=False) + '\n').encode())
                        shutil.copyfileobj(source, output)
    if missing:
        raise RuntimeError(vm['name'] + ': missing ' + ', '.join(missing))


def run_trial(vms, builder, art, protocol, load, repetition, load_index, session, common, client_art=None):
    label = trial_label(protocol, load, repetition, load_index)
    folder = session / label
    folder.mkdir(parents=True)
    logs_folder = Path(RESULTS_DIR).expanduser().resolve() / label
    logs_folder.mkdir(parents=True, exist_ok=True)
    # A rerun replaces this exact trial. Do not leave old client data behind if
    # the new run fails before collecting logs, and never touch other trials.
    for pattern in ('node-*/stellar-core.log', 'client-*/client.log'):
        for old_log in logs_folder.glob(pattern):
            old_log.unlink()
    # Each retry gets a new directory; preserve the failed attempt logs.
    trial = REMOTE_SESSION + "/runs/" + label + "/attempt-" + uuid.uuid4().hex
    client_art = client_art or art
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load['active_clients']]
    config = {**common, **load, "protocol": protocol, "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    states = []
    cleanup_errors = []
    timings = {}
    setup_start = time.monotonic()
    print(f'Starting {label}: setup, then {CLIENT_START_LEAD_SEC}s scheduled client lead.', flush=True)
    try:
        parallel(stop_vm_processes, vms)
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder, protocol)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join(runtime_command([art + "/stellar-core", "new-db", "--conf",
                                            root + "/stellar-core.cfg"], protocol)),
                log=folder / f"node-{vm['index']}-init.log")
            launch_process(vm, root, runtime_command(
                [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"], protocol),
                           art + "/lib",
                           "console.log")
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        last_connectivity = None
        while True:
            status = parallel(lambda v: process_state(v, trial + f"/nodes/node{v['index']}", check_ready=True), nodes)
            if any(s['done'] or s.get('error') for s in status):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise RuntimeError('Replica exited during startup: ' + str(result['replica_outcomes']))
            connectivity = (sum(s['ready'] for s in status),
                            min(s.get('connected_peers', 0) for s in status))
            if connectivity != last_connectivity:
                print(f"Connectivity: {connectivity[0]}/{len(nodes)} replicas fully connected; "
                      f"minimum {connectivity[1]}/{len(nodes)-1} authenticated peers", flush=True)
                last_connectivity = connectivity
            if all(s['ready'] for s in status) and status[0]['listener']:
                break
            if time.monotonic() >= deadline:
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise TimeoutError("Full peer connectivity/client listener timed out: " +
                                   str(result['replica_outcomes']))
            time.sleep(3)
        start_at = time.time() + CLIENT_START_LEAD_SEC
        timings['setup_sec'] = round(time.monotonic() - setup_start, 3)
        config['scheduled_client_start'] = start_at
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            launch_process(vm, trial + f"/client-{vm['index']}",
                runtime_command([client_art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                 "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                 str(SEND_INTERVAL_US), str(load['client_threads'])], protocol),
                client_art + "/lib", "client.log", start_at=start_at)
        parallel(start_client, clients)
        if time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        workload_end = start_at + CLIENT_DURATION_SEC
        while time.time() < workload_end:
            # Poll replicas and clients in one batch; neither group depends on the other.
            observed = parallel(lambda pair: process_state(*pair), owned)
            replica_states, states = observed[:len(nodes)], observed[len(nodes):]
            if any(s['done'] or s.get('error') for s in replica_states):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], replica_states))
                raise RuntimeError('Replica exited during workload: ' + str(result['replica_outcomes']))
            if any(s.get('error') for s in states):
                raise RuntimeError('Client process failed: ' + str(states))
            if all(s['done'] for s in states):
                break
            time.sleep(min(3, max(0, workload_end - time.time())))
        result['client_outcomes'] = states
        if any('started_at' not in s for s in states):
            raise RuntimeError('A client never started: ' + str(states))
        starts = [s['started_at'] for s in states]
        result['client_start_skew_sec'] = max(starts) - min(starts)
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or (s['done'] and s['returncode'] != 0) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'ok'
        result['stop_reason'] = 'workload_duration_reached' if time.time() >= workload_end else 'clients_completed'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        stop_start = time.monotonic()
        def stop(pair):
            vm, root = pair
            try:
                stop_vm_processes(vm)
            except Exception as exc:
                cleanup_errors.append(f"{vm['name']}: {exc}")
        parallel(stop, owned)
        timings['stop_sec'] = round(time.monotonic() - stop_start, 3)
        collect_start = time.monotonic()
        def collect(pair):
            vm, root = pair
            local = logs_folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            collect_trial_logs(vm, root, local, config)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        timings['collect_sec'] = round(time.monotonic() - collect_start, 3)
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        if result['status'] in ('ok', 'drain_timeout'):
            def count_commits(vm):
                with (logs_folder / f"node-{vm['index']}" / 'stellar-core.log').open(errors='replace') as log:
                    return sum(line.count(tag) for line in log for tag in COMMIT_TAGS[protocol])
            counts = parallel(count_commits, nodes)
            result['replica_commits'] = counts
            if not all(counts):
                result.update(status='failed', error='A replica logged no commits')
        result['timings_sec'] = timings
        save_json(folder / "result.json", result)
        print(f"Cleanup {label}: stop={timings['stop_sec']:.1f}s, logs={timings['collect_sec']:.1f}s", flush=True)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError(result.get('error', 'Trial cleanup failed') + '\nCleanup: ' +
                           '; '.join(cleanup_errors) + '\nLogs: ' + str(logs_folder))
    return result


def trial_label(protocol, load, repetition, index):
    return (f"n{NUM_NODES}/{protocol}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def stop_vm_processes(vm, *, include_builds=False):
    # These are dedicated experiment VMs. Restrict cleanup to the SSH user's processes.
    # Bracketed dots keep pkill from matching the cleanup command itself.
    pattern = re.escape(REMOTE_ROOT) + r'/[^ ]*/(launch[.]sh|worker[.]py supervise )'
    script = """
kill_matching() {
    pkill -u "$(id -u)" "$@" || { rc=$?; [ "$rc" -eq 1 ]; }
}
kill_matching -TERM -f PATTERN
kill_matching -TERM -x stellar-core
kill_matching -TERM -x shab_client
if pgrep -x stellar-core >/dev/null || pgrep -x shab_client >/dev/null; then
    sleep 1
fi
kill_matching -KILL -x stellar-core
kill_matching -KILL -x shab_client
kill_matching -KILL -f PATTERN
""".replace('PATTERN', shlex.quote(pattern))
    if include_builds:
        # Kill old notebook build supervisors first, so they cannot start another
        # make/compiler after cleanup. The bracket prevents matching this script.
        # Dedicated VMs may have orphaned children after a kernel/SSH interruption.
        build_cleanup = """
kill_matching -KILL -f '[B]UILD_TIMINGS_JSON='
kill_matching -KILL -x '(make|gmake|ninja|ninja-build|cmake|cargo)(-[0-9.]+)?'
kill_matching -KILL -x '(gcc|g[+][+]|cc|c[+][+]|clang|clang[+][+]|cc1|cc1plus|collect2|lto-wrapper|lto1|as|ld|ld[.]bfd|ld[.]gold|ld[.]lld|lld|rustc|rustdoc|build-script-bu)(-[0-9.]+)?'
"""
        # Insert after the shell helper, before stopping replicas and clients.
        script = script.replace("\n}\n", "\n}\n" + build_cleanup, 1)
    ssh(vm, script, timeout=30)


def prepare_build(vm, sha, folder):
    # Update the existing checkout once. Never copy it, clean it, or reinstall dependencies.
    script = f"""
cd {shlex.quote(IMAGE_REPO)}
test -d .git
test -f Makefile
if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
    export GIT_TERMINAL_PROMPT=0
    # The sweep itself edits this header; clear those generated flags before pulling.
    if ! git diff --quiet HEAD -- src/overlay/experiments/ExperimentConfig.h; then
        git restore --source=HEAD -- src/overlay/experiments/ExperimentConfig.h
    fi
    git pull --ff-only origin {shlex.quote(sha)}
    if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
        echo "Existing checkout is ahead of the session commit; refusing to mix source versions." >&2
        exit 1
    fi
fi
printf 'Using existing configured checkout: %s\\n' "$PWD"
"""
    ssh(vm, script, log=folder / 'prepare-build.log', timeout=600)


def finish_session(session):
    print('Finished:', Path(RESULTS_DIR).expanduser().resolve(),
          '\nRun the Main num nodes experiment plotting cell in PostProcess.ipynb with the same RESULTS_DIR.')
    saved = json.loads((session / 'fleet.json').read_text()) if (session / 'fleet.json').exists() else {'vms': []}
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [v for v in saved['vms'] if (v['name'], v['zone']) in existing]
        print('Deleting remaining replica/client VMs:', ', '.join(v['name'] for v in remaining), flush=True)
        parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
                 remaining)
    else:
        print('VMs kept running for reuse (DELETE_ALL_AFTER_FINAL_RUN=False).')


def validate_size_settings():
    if type(RESUME_SESSION) is not bool:
        raise ValueError('RESUME_SESSION must be True or False')
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if (not NUM_NODES_LIST or any(type(n) is not int or n < 4 for n in NUM_NODES_LIST)
            or any(a <= b for a, b in zip(NUM_NODES_LIST, NUM_NODES_LIST[1:]))):
        raise ValueError('NUM_NODES_LIST must contain distinct sizes >=4 in strictly descending order')
    if len(REPLICA_ZONES) != NUM_NODES_LIST[0]:
        raise ValueError('Specify one REPLICA_ZONES entry per replica in the largest fleet')
    if 'forget_it' in PROTOCOLS and any((n - 1) % 3 for n in NUM_NODES_LIST):
        raise ValueError('Forget-IT requires n=3f+1: use [49, 31, 16, 7, 4] for the common sweep')
    if not PROTOCOLS or len(set(PROTOCOLS)) != len(PROTOCOLS) or set(PROTOCOLS) - COMMIT_TAGS.keys():
        raise ValueError('Choose distinct custom-client protocols; SCP uses a different workload')
    if set(PROTOCOLS) - PROTOCOL_LOADS.keys():
        raise ValueError('Supply PROTOCOL_LOADS for every selected protocol')
    if not (0 <= MEASUREMENT_START_SEC < MEASUREMENT_END_SEC <= CLIENT_DURATION_SEC):
        raise ValueError('Invalid measurement window')
    counts = (REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE, MAX_PARALLEL_WORKERS)
    if any(type(v) is not int or v <= 0 for v in counts):
        raise ValueError('Repetitions, build jobs, batch size, and parallel workers must be positive integers')
    if min(ITHS_BLOG_DELTA_MS, FORGET_IT_DELTA_MS) <= 0:
        raise ValueError('Delta must be positive')
    if not re.fullmatch(r'WORKLOAD_(W|R0)', WORKLOAD):
        raise ValueError('Use write-only transactions: WORKLOAD_W or WORKLOAD_R0')
    for protocol in PROTOCOLS:
        load = PROTOCOL_LOADS[protocol]
        if (set(load) != {'active_clients', 'client_threads', 'max_in_flight'}
                or any(type(v) is not int or v <= 0 for v in load.values())):
            raise ValueError(f'Invalid load for {protocol}: {load}')
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError(f'{protocol}: insufficient client VMs or concurrency for one full batch')


def shrink_fleet(vms, next_size, session):
    # Client VMs have separate names/indices, so every client survives all shrinks.
    surplus = [v for v in vms if v['role'] == 'node' and v['index'] > next_size]
    print(f'Deleting {len(surplus)} surplus replicas before n={next_size}: ' +
          ', '.join(v['name'] for v in surplus), flush=True)
    parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
             surplus)
    retained = [v for v in vms if v['role'] == 'client' or v['index'] <= next_size]
    saved = json.loads((session / 'fleet.json').read_text())
    save_json(session / 'fleet.json', {**saved, 'vms': retained})
    return retained


def cleanup_size_fleet(vms):
    # Deletion may have partially succeeded before an exception/interrupt.
    # Do not attempt SSH to deleted VMs, and still try every surviving VM.
    existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                json.loads(cloud('instances', 'list', '--format=json'))}
    surviving = [v for v in vms if (v['name'], v['zone']) in existing]
    parallel(lambda vm: stop_vm_processes(vm, include_builds=True), surviving)


def main_gcp_sizes(work_dir):
    global REMOTE_SESSION, REMOTE_BUILD, NUM_NODES
    validate_size_settings()           # Validate everything before creating/deleting VMs.
    NUM_NODES = NUM_NODES_LIST[0]
    repo = REPO.expanduser().resolve()
    sha = header = None
    if not RESUME_SESSION:
        sha = command(['git', '-C', repo, 'rev-parse', (SOURCE_COMMIT or 'HEAD') + '^{commit}']).strip()
        header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
        require_runtime_config(header)
    session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    session = Path(work_dir)
    session.mkdir(parents=True, exist_ok=True)
    REMOTE_SESSION = REMOTE_ROOT + '/' + session_id
    REMOTE_BUILD = IMAGE_REPO
    common = {'schema_version': 1, 'experiment': 'num_nodes', 'session': session_id,
              'source_commit': sha, 'workload': WORKLOAD, 'batch_size': SERVER_BATCH_SIZE,
              'client_duration_sec': CLIENT_DURATION_SEC,
              'measurement_start_sec': MEASUREMENT_START_SEC, 'measurement_end_sec': MEASUREMENT_END_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'iths_blog_delta_ms': ITHS_BLOG_DELTA_MS, 'forget_it_delta_ms': FORGET_IT_DELTA_MS,
              'shabdiz_fast_path_only': SHABDIZ_FAST_PATH_ONLY, 'forced_collection': False,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': 0}
    manifest = {**common, 'protocols': PROTOCOLS, 'protocol_loads': PROTOCOL_LOADS,
                'num_nodes_list': NUM_NODES_LIST, 'repetitions': REPETITIONS,
                'instance_prefix': INSTANCE_PREFIX, 'resume_session': RESUME_SESSION,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN}
    save_json(session / 'session.json', manifest)
    print('Results:', Path(RESULTS_DIR).expanduser().resolve(), flush=True)
    print('Replica counts:', NUM_NODES_LIST, '; additional client VMs:', len(CLIENT_ZONES), flush=True)
    print('Loads:', {p: math.prod(PROTOCOL_LOADS[p].values()) for p in PROTOCOLS}, flush=True)
    total = len(NUM_NODES_LIST) * len(PROTOCOLS) * REPETITIONS
    print(f'{total} trials; {total * CLIENT_DURATION_SEC / 3600:.2f} hours of workload, '
          'plus startup/builds/log collection. pkill at workload end.', flush=True)
    results = []
    save_json(session / 'results.json', results)
    vms = fleet(session)               # Create ONLY ONCE, with the largest replica count.
    try:
        parallel(wait_ssh, vms)
        common['resolved_image'] = json.loads((session / 'fleet.json').read_text())['image']
        print('Stopping stale replicas, clients, and builds on the largest fleet...', flush=True)
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
        def initialize_vm(vm):
            sync = ssh(vm, shlex.join(['mkdir', '-p', REMOTE_SESSION]) + '\n' +
                       'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before scheduled clients")
        parallel(initialize_vm, vms)
        builder = next(v for v in vms if v['role'] == 'node')
        if RESUME_SESSION:
            def existing_source(vm):
                check = 'test -f shab_client.cpp' if vm['role'] == 'client' else 'test -f Makefile'
                return ssh(vm, 'cd ' + shlex.quote(IMAGE_REPO) + '\n' + check + '\ngit rev-parse HEAD').strip()
            sources = parallel(existing_source, vms)
            if len(set(sources)) != 1:
                raise RuntimeError('VM source revisions differ; rerun with RESUME_SESSION=False')
            sha = sources[0]
            header = ssh(builder, 'cd ' + shlex.quote(IMAGE_REPO) +
                         '\ngit show HEAD:src/overlay/experiments/ExperimentConfig.h')
            require_runtime_config(header)
        else:
            def prepare_vm(vm):
                folder = session / 'preparation' / vm['name']
                folder.mkdir(parents=True, exist_ok=True)
                prepare_build(vm, sha, folder)
            print('Preparing the pinned source once on every VM...', flush=True)
            parallel(prepare_vm, vms)
        common['source_commit'] = manifest['source_commit'] = sha
        save_json(session / 'session.json', manifest)
        print('Source:', sha, flush=True)
        def build_common(vm):
            folder = session / 'builds' / vm['name']
            if vm['role'] == 'node':
                build_replica(vm, header, folder)
            else:
                build_client(vm, header, folder)
        print(f'Checking common binaries once, make -j{BUILD_JOBS} per replica...', flush=True)
        parallel(build_common, vms)
        art, client_art = IMAGE_REPO + '/src', IMAGE_REPO
        for size_index, num_nodes in enumerate(NUM_NODES_LIST):
            NUM_NODES = num_nodes      # Config membership only; no header edits/recompilation.
            nodes = [v for v in vms if v['role'] == 'node']
            if len(nodes) != num_nodes:
                raise RuntimeError(f'Expected {num_nodes} replicas, got {len(nodes)}')
            run_common = {**common, 'num_nodes': num_nodes,
                          'replica_zones': [v['zone'] for v in nodes]}
            print(f'\n=== {num_nodes} replicas + {len(CLIENT_ZONES)} client VMs ===', flush=True)
            size_results = []
            for protocol in PROTOCOLS:
                load = dict(PROTOCOL_LOADS[protocol])
                print(f'{protocol}: {math.prod(load.values())} concurrent requests; reusing binaries.', flush=True)
                for repetition in range(1, REPETITIONS + 1):
                    result = run_trial(vms, builder, art, protocol, load, repetition, 1,
                                       session, run_common, client_art)
                    result['num_nodes'] = num_nodes
                    size_results.append(result)
                    results.append(result)
                    save_json(session / 'results.json', results)
            failed = [r for r in size_results if r['status'] != 'ok']
            if failed:
                raise RuntimeError(f'{len(failed)} trials failed at n={num_nodes}; '
                                   'logs collected, fleet kept for inspection; smaller sizes were not started')
            if size_index + 1 < len(NUM_NODES_LIST):
                vms = shrink_fleet(vms, NUM_NODES_LIST[size_index + 1], session)
    finally:
        cleanup_size_fleet(vms)
    finish_session(session)
    return Path(RESULTS_DIR).expanduser().resolve()


def run_gcp_sizes():
    validate_size_settings()
    Path(RESULTS_DIR).expanduser().resolve().mkdir(parents=True, exist_ok=True)
    # Temporary setup/status files disappear; retain only replica/client logs and optional build logs.
    with tempfile.TemporaryDirectory(prefix='stellar-gcp-size-') as work_dir:
        return main_gcp_sizes(work_dir)


GCP_SIZE_SESSION = run_gcp_sizes()


# Failure Experiment

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-c']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-c', 'us-central1-c', 'us-central1-c', 'us-central1-c']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [16]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 180
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-c"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     do_compile_this_run = (not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         # results = run_parallel(
#         #     compile_stellar,
#         #     range(num_nodes + n_clients),
#         #     max_workers=min(48, num_nodes + n_clients)
#         # )


        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     # if num_nodes >= 2:
#     #     target_file = "../stellar-private/node2/stellar-core.cfg"
#     #     line_to_add = "MEMORY_PROF=true"

#     #     subprocess.call(
#     #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#     #         shell=True
#     #     )

#     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         # time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         time.sleep(100)

#         results = run_parallel(
#             kill_stellar_private,
#             [4],
#             max_workers=min(48, num_nodes + n_clients)
#         )

        
#         time.sleep(20)

        


#         results = run_parallel(
#             kill_stellar_private,
#             [5],
#             max_workers=min(48, num_nodes + n_clients)
#         )

        
#         time.sleep(20)


#         results = run_parallel(
#             kill_stellar_private,
#             [6],
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         time.sleep(20)
        
#         results = run_parallel(
#             kill_stellar_private,
#             [7],
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         time.sleep(20)
        
#         results = run_parallel(
#             kill_stellar_private,
#             [8],
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         time.sleep(80)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"Failure_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

#     # -------------------------------------------------------------------------
#     # After this run, delete only the machines that the next smaller run will
#     # not need. Example:
#     #   after 48-node run, keep 000..032 and delete 033..048
#     #   after 32-node run, keep 000..016 and delete 017..032
#     #   after 16-node run, keep 000..008 and delete 009..016
#     #   after 8-node run,  keep 000..004 and delete 005..008
#     # Final run optionally deletes everything.
#     # -------------------------------------------------------------------------
#     if DELETE_UNUSED_AFTER_EACH_RUN:
#         instances = fetch_existing_instances()

#         if run_idx + 1 < len(NUM_NODES_LIST):
#             next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#             keep_count = next_num_nodes + n_clients

#             instances_to_delete = [
#                 (name, inst_zone)
#                 for name, inst_zone in instances
#                 if parse_tsm_index(name) >= keep_count
#             ]

#             print(
#                 f"\n➡ After num_nodes={num_nodes}, next run needs "
#                 f"indices 0..{keep_count - 1}. Deleting higher-index VMs:"
#             )
#         elif DELETE_ALL_AFTER_FINAL_RUN:
#             instances_to_delete = instances

#             print("\n➡ Final run complete. Deleting all remaining tsm-sc-* VMs:")
#         else:
#             instances_to_delete = []

#             print("\n✔ Final run complete. Leaving remaining tsm-sc-* VMs running.")

#         for name, inst_zone in instances_to_delete:
#             print(f"  - {name} ({inst_zone})")

#         if instances_to_delete:
#             run_parallel(
#                 delete_instance,
#                 instances_to_delete,
#                 max_workers=min(32, len(instances_to_delete))
#             )
#             print(f"\n🧹 Deleted {len(instances_to_delete)} unneeded instance(s) after num_nodes={num_nodes}.\n")
#         else:
#             print("\n✔ No unneeded instances to delete after this run.\n")


# Collection vs Collection Rounds Experiment

In [ ]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
import re
from pathlib import Path


# ============================================================
# Helpers
# ============================================================

def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


def fetch_existing_instances(project):
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''

    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []

    for line in output.splitlines():
        if line.strip():
            name, inst_zone = line.split()
            instances.append((name, inst_zone))

    return instances


def delete_instance(project, instance):
    name, inst_zone = instance

    cmd = f'''
    gcloud compute instances delete {name} \
        --zone={inst_zone} \
        --project={project} \
        --quiet
    '''

    print(f"🗑️ Deleting {name} in {inst_zone}")
    return subprocess.call(cmd, shell=True)


def delete_all_tsm_instances(project):
    instances = fetch_existing_instances(project)

    print("\n➡ Existing instances to delete:")
    for name, inst_zone in instances:
        print(f"  - {name} ({inst_zone})")

    if instances:
        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances,
            max_workers=min(32, len(instances)),
        )
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


# ============================================================
# Experiment config
# ============================================================

# latencies: 50, 90, 150, 210
default_region = ["us-west1-b"]
# regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
regions = ["us-west1-b", "us-west1-b", "us-west1-b", "us-west1-b"]

zone_no = 0

# System size for this failure / collection-window experiment.
NUM_NODES_LIST = [16]

# Use 1 extra 2-core machine as client machine.
# For num_nodes = N, client is tsm-sc-N.
n_clients = 1

# Collection-round sweep.
COLLECT_ATTEMPT_POINTS = [300]

# Source file to patch before each subrun.
OVERLAY_MANAGER_IMPL_PATH = Path("src/overlay/OverlayManagerImpl.cpp")

# Start clean only once, before the first run.
DELETE_BEFORE_FIRST_RUN = True

# Delete VMs after each MAX_COLLECT_ATTEMPTS subrun.
DELETE_UNUSED_AFTER_EACH_RUN = True

# Delete remaining VMs after final run.
# This is mostly redundant if DELETE_UNUSED_AFTER_EACH_RUN=True.
DELETE_ALL_AFTER_FINAL_RUN = True

# IMPORTANT:
# Code changes across collection-round settings, so we must compile every subrun.
COMPILE_ONLY_FIRST_RUN = False

MAX_NUM_NODES = max(NUM_NODES_LIST)

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed offered load.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]


# ============================================================
# Local source patching
# ============================================================

def patch_max_collect_attempts(max_collect_attempts):
    text = OVERLAY_MANAGER_IMPL_PATH.read_text()

    pattern = r"static\s+constexpr\s+uint64_t\s+MAX_COLLECT_ATTEMPTS\s*=\s*\d+\s*;"
    replacement = (
        f"static constexpr uint64_t MAX_COLLECT_ATTEMPTS = {max_collect_attempts};"
    )

    new_text, count = re.subn(
        pattern,
        replacement,
        text,
        flags=re.MULTILINE,
    )

    if count != 1:
        raise RuntimeError(
            f"Expected to replace exactly one MAX_COLLECT_ATTEMPTS line in "
            f"{OVERLAY_MANAGER_IMPL_PATH}, but replaced {count}"
        )

    OVERLAY_MANAGER_IMPL_PATH.write_text(new_text)

    print(
        f"Updated {OVERLAY_MANAGER_IMPL_PATH}: "
        f"MAX_COLLECT_ATTEMPTS = {max_collect_attempts}"
    )


def push_collect_patch_to_git(max_collect_attempts):
    cmd = (
        f'git add {OVERLAY_MANAGER_IMPL_PATH}; '
        f'git commit -m "set collect attempts {max_collect_attempts}" || true; '
        f'git push'
    )

    return subprocess.call(cmd, shell=True)


# ============================================================
# Main experiment loop
# ============================================================

for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
    for collect_idx, max_collect_attempts in enumerate(COLLECT_ATTEMPT_POINTS):

        project = "research-488322"
        zone = "us-west1-b"
        machine_type = "e2-standard-2"
        image_family = "tsm-sc-family"
        subnet = "default"
        gcp_username = "tejas"

        print("\n" + "#" * 100)
        print(
            f"Starting experiment for num_nodes={num_nodes}, "
            f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}"
        )
        print("#" * 100)

        def get_zone_for_instance(i):
            # Use MAX_NUM_NODES so zone assignment stays stable.
            if i < int(MAX_NUM_NODES / 2):
                return default_region[0]
            else:
                return regions[zone_no]

        def parse_tsm_index(name):
            return int(name.rsplit("-", 1)[1])

        # -----------------------------------------------------------------
        # Delete existing tsm-sc-* instances only before the first subrun,
        # if enabled.
        # -----------------------------------------------------------------
        if DELETE_BEFORE_FIRST_RUN and run_idx == 0 and collect_idx == 0:
            print("\n🧹 Cleaning before first run...")
            delete_all_tsm_instances(project)

        # -----------------------------------------------------------------
        # Create only missing replica/client machines for this subrun.
        # -----------------------------------------------------------------
        existing_instances = fetch_existing_instances(project)
        existing_names = {name for name, _ in existing_instances}

        commands = []
        newly_created_indices = []

        # Create replica nodes only if they do not already exist.
        for i in range(num_nodes):
            instance_name = f"tsm-sc-{i:03}"

            if instance_name in existing_names:
                print(f"✔ Reusing existing replica {instance_name}")
                continue

            inst_zone = get_zone_for_instance(i)

            cmd = f'''
            gcloud compute instances create {instance_name} \
                --project={project} \
                --zone={inst_zone} \
                --machine-type={machine_type} \
                --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
                --can-ip-forward \
                --maintenance-policy=MIGRATE \
                --provisioning-model=STANDARD \
                --service-account=254510644191-compute@developer.gserviceaccount.com \
                --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
                --tags=http-server,https-server \
                --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
                --no-shielded-secure-boot \
                --shielded-vtpm \
                --shielded-integrity-monitoring \
                --labels=goog-ec-src=vm_add-gcloud \
                --reservation-affinity=any
            '''
            commands.append(cmd.strip())
            newly_created_indices.append(i)

        # Create client machine only if it does not already exist.
        for i in range(n_clients):
            client_idx = num_nodes + i
            instance_name = f"tsm-sc-{client_idx:03}"

            if instance_name in existing_names:
                print(f"✔ Reusing existing client {instance_name}")
                continue

            inst_zone = get_zone_for_instance(client_idx)

            cmd = f'''
            gcloud compute instances create {instance_name} \
                --project={project} \
                --zone={inst_zone} \
                --machine-type={machine_type} \
                --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
                --can-ip-forward \
                --maintenance-policy=MIGRATE \
                --provisioning-model=STANDARD \
                --service-account=254510644191-compute@developer.gserviceaccount.com \
                --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
                --tags=http-server,https-server \
                --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
                --no-shielded-secure-boot \
                --shielded-vtpm \
                --shielded-integrity-monitoring \
                --labels=goog-ec-src=vm_add-gcloud \
                --reservation-affinity=any
            '''
            commands.append(cmd.strip())
            newly_created_indices.append(client_idx)

        if commands:
            run_parallel(
                run_command,
                commands,
                max_workers=min(48, len(commands)),
            )

            print("All missing instances launched.")

            # Give GCP/SSH a little time after VM creation.
            time.sleep(30)
        else:
            print("✔ All required instances already exist; no VM creation needed.")

        # -----------------------------------------------------------------
        # Get sorted node and client IPs.
        # tsm_ips.txt must include only replica IPs, not clients.
        # -----------------------------------------------------------------
        ip_cmd = f'''
        gcloud compute instances list \
            --project={project} \
            --filter="name~'^tsm-sc-'" \
            --sort-by=name \
            --format="value(name,zone,networkInterfaces[0].networkIP)"
        '''

        ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

        instance_records = []

        for line in ip_output.splitlines():
            if line.strip():
                name, inst_zone, ip = line.split()
                idx = int(name.rsplit("-", 1)[1])
                instance_records.append((idx, name, inst_zone, ip))

        instance_records.sort()

        node_records = [r for r in instance_records if r[0] < num_nodes]
        client_records = [
            r for r in instance_records
            if num_nodes <= r[0] < num_nodes + n_clients
        ]

        if len(node_records) != num_nodes:
            raise RuntimeError(
                f"Expected {num_nodes} replica nodes, but found "
                f"{len(node_records)}: {node_records}"
            )

        if len(client_records) != n_clients:
            raise RuntimeError(
                f"Expected {n_clients} client nodes, but found "
                f"{len(client_records)}: {client_records}"
            )

        iplist = [r[3] for r in node_records]

        with open("tsm_ips.txt", "w") as f:
            for ip in iplist:
                f.write(ip + "\n")

        print("🎯 Node IPs:", iplist)
        print("🎯 Client instances:", client_records)

        node1_ip = iplist[0]
        print(f"Clients will connect to leader/node1 at: {node1_ip}")

        # -----------------------------------------------------------------
        # Remote helper functions.
        # -----------------------------------------------------------------

        def kill_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            remote_command = """\
sudo pkill -9 stellar-core || true; \
sudo pkill -9 shab_client || true; \
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")
            output = subprocess.call(command, shell=True)
            print(f"Return code for tsm-sc-{i:03}: {output}")
            return output

        def git_pull_stellar(i):
            inst_zone = get_zone_for_instance(i)

            command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

            print(command)
            output = subprocess.call(command, shell=True)
            print(output)
            return output

        def compile_stellar(i):
            inst_zone = get_zone_for_instance(i)

            command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j4; \
cd; \
sudo rm -rf stellar-private"'''

            print(command)
            output = subprocess.call(command, shell=True)
            print(output)
            return output

        def clean_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            remote_command = """\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")
            output = subprocess.call(command, shell=True)
            print(f"Return code for tsm-sc-{i:03}: {output}")
            return output

        def copy_folder_to_instance(
            i,
            source_folder="/home/tejas/stellar-private",
            destination_path="/home/tejas/stellar-private",
        ):
            inst_zone = get_zone_for_instance(i)
            instance_name = f"tsm-sc-{i:03}"

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

            print(f"Executing command for {instance_name}: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Command for {instance_name} finished with exit code: {output}")

            return (instance_name, output)

        def run_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            node_number = i + 1
            instance_name = f"tsm-sc-{i:03}"

            remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Return code for {instance_name}: {output}")
            return output

        def run_stellar_client(i, client_max_in_flight, client_threads):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"
            client_id = i - num_nodes

            remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing client command: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Return code for {instance_name}: {output}")
            return output

        # -----------------------------------------------------------------
        # Generate stellar-private configs locally using only replica IPs.
        # -----------------------------------------------------------------
        stellar_private_path = Path("../stellar-private")
        if stellar_private_path.exists():
            shutil.rmtree(stellar_private_path)
        stellar_private_path.mkdir()

        subprocess.call(
            "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
            shell=True,
        )

        subprocess.call(
            "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
            "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
            shell=True,
        )

        # Enable custom message only on leader.
        line_to_add = "SEND_CUSTOM_MESSAGE=true"
        target_file = "../stellar-private/node1/stellar-core.cfg"

        subprocess.call(
            f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
            shell=True,
        )

        print("\n" + "*" * 100)
        print(f"Preparing subrun with MAX_COLLECT_ATTEMPTS={max_collect_attempts}")
        print("*" * 100)

        # Patch local source.
        patch_max_collect_attempts(max_collect_attempts)

        # Push patched source.
        push_collect_patch_to_git(max_collect_attempts)

        # Stop any old processes.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Pull patched code on active VMs.
        results = run_parallel(
            git_pull_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Compile patched code on active VMs.
        results = run_parallel(
            compile_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # -----------------------------------------------------------------
        # Throughput/latency loop.
        #
        # IMPORTANT:
        # This is now inside the MAX_COLLECT_ATTEMPTS loop.
        # So every collect-attempt value gets its own actual run.
        # -----------------------------------------------------------------
        for load in LOAD_POINTS:
            active_clients = load["active_clients"]
            client_threads = load["client_threads"]
            client_max_in_flight = load["max_in_flight"]

            if active_clients > n_clients:
                raise RuntimeError(
                    f"active_clients={active_clients} exceeds n_clients={n_clients}"
                )

            total_client_threads = active_clients * client_threads
            aggregate_max_in_flight = (
                active_clients * client_threads * client_max_in_flight
            )

            run_label = (
                f"clients_{active_clients}_threads_{client_threads}_"
                f"inflight_{client_max_in_flight}_"
                f"total_threads_{total_client_threads}_"
                f"total_inflight_{aggregate_max_in_flight}"
            )

            print("\n" + "=" * 80)
            print(
                f"🚀 Starting run: num_nodes={num_nodes}, "
                f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, {run_label}"
            )
            print("=" * 80)

            # Clean remote stellar-private.
            results = run_parallel(
                clean_stellar_private,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # Copy fresh stellar-private config to all active machines.
            results = run_parallel(
                copy_folder_to_instance,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # Kill old processes on all active machines.
            results = run_parallel(
                kill_stellar_private,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # Start consensus replicas.
            results = run_parallel(
                run_stellar_private,
                range(num_nodes),
                max_workers=min(48, num_nodes),
            )

            print(results)
            print("All Stellar nodes should be starting in the background.")

            # Give nodes time to authenticate and start the client listener.
            time.sleep(40)

            # Start only required client VMs for this load point.
            active_client_indices = [
                num_nodes + j for j in range(active_clients)
            ]

            results = run_parallel(
                lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
                active_client_indices,
                max_workers=active_clients,
            )

            print(results)
            print(
                f"Started {active_clients} client VM(s), "
                f"each with {client_threads} client threads. "
                f"Total client threads = {total_client_threads}. "
                f"Aggregate max in-flight = {aggregate_max_in_flight}."
            )

            # Let the system run.
            time.sleep(CLIENT_WAIT_AFTER_START_SEC)

            # Stop all nodes and clients.
            results = run_parallel(
                kill_stellar_private,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # -------------------------------------------------------------
            # Save logs.
            # -------------------------------------------------------------
            remote_base_folder = "/home/tejas/stellar-private"

            local_base_destination = (
                "/home/tejas/work/experiments/shabdiz/"
                + f"Collection_{max_collect_attempts}_"
                + f"nodes_{num_nodes}_{run_label}"
            )

            Path(local_base_destination).mkdir(parents=True, exist_ok=True)

            # Save run metadata.
            with open(Path(local_base_destination) / "run_config.txt", "w") as f:
                f.write(f"num_nodes={num_nodes}\n")
                f.write(f"max_collect_attempts={max_collect_attempts}\n")
                f.write(f"force_collect_after_sec=100\n")
                f.write(f"active_clients={active_clients}\n")
                f.write(f"client_threads_per_vm={client_threads}\n")
                f.write(f"total_client_threads={total_client_threads}\n")
                f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
                f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
                f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
                f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
                f.write(f"leader_ip={node1_ip}\n")
                f.write(f"machine_type={machine_type}\n")
                f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

            def copy_folder_from_instance(i):
                inst_zone = get_zone_for_instance(i)

                instance_name = f"tsm-sc-{i:03}"

                node_number = i + 1
                node_folder = f"node{node_number}"

                remote_source_path = posixpath.join(remote_base_folder, node_folder)

                local_destination_path = Path(local_base_destination) / instance_name
                local_destination_path.mkdir(parents=True, exist_ok=True)

                remote_source = f"{instance_name}:{remote_source_path}"

                command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

                print(
                    f"Executing command to copy {node_folder} "
                    f"from {instance_name}: {command}"
                )

                output = subprocess.call(command, shell=True)

                print(f"Copy from {instance_name} finished with exit code: {output}")

                return (instance_name, output)

            # Copy all replica logs.
            # FIX: was range(3), which copied only nodes 0, 1, 2.
            node_copy_results = run_parallel(
                copy_folder_from_instance,
                range(3),
                max_workers=3,
            )

            def copy_client_log(i):
                inst_zone = get_zone_for_instance(i)

                instance_name = f"tsm-sc-{i:03}"
                client_id = i - num_nodes

                remote_source = (
                    f"{instance_name}:/home/tejas/stellar-private/"
                    f"stellar-client-{client_id}.log"
                )

                local_destination_path = Path(local_base_destination)
                local_destination_path.mkdir(parents=True, exist_ok=True)

                command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

                print(f"Copying client log from {instance_name}...")

                output = subprocess.call(command, shell=True)

                print(f"Copy finished with exit code: {output}")

                return (instance_name, output)

            client_copy_results = run_parallel(
                copy_client_log,
                active_client_indices,
                max_workers=active_clients,
            )

            print("\n--- Summary of Download Results ---")
            print("Node log copies:", node_copy_results)
            print("Client log copies:", client_copy_results)
            print(f"Saved run to: {local_base_destination}")

        # -----------------------------------------------------------------
        # Delete instances after this MAX_COLLECT_ATTEMPTS subrun.
        #
        # IMPORTANT:
        # This is inside the collect-attempt loop, so after Collection_50
        # finishes, VMs are deleted before Collection_300 starts.
        # -----------------------------------------------------------------
        if DELETE_UNUSED_AFTER_EACH_RUN:
            print(
                "\n🧹 Deleting VMs after completed subrun: "
                f"num_nodes={num_nodes}, MAX_COLLECT_ATTEMPTS={max_collect_attempts}"
            )
            delete_all_tsm_instances(project)


# ============================================================
# Final cleanup
# ============================================================

if DELETE_ALL_AFTER_FINAL_RUN and not DELETE_UNUSED_AFTER_EACH_RUN:
    print("\n🧹 Final cleanup after all runs...")
    delete_all_tsm_instances(project)

# Collection vs num nodes

In [ ]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
import re
from pathlib import Path


# ============================================================
# Helpers
# ============================================================

def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


def fetch_existing_instances(project):
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''

    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []

    for line in output.splitlines():
        if line.strip():
            name, inst_zone = line.split()
            instances.append((name, inst_zone))

    return instances


def parse_tsm_index(name):
    return int(name.rsplit("-", 1)[1])


def delete_instance(project, instance):
    name, inst_zone = instance

    cmd = f'''
    gcloud compute instances delete {name} \
        --zone={inst_zone} \
        --project={project} \
        --quiet
    '''

    print(f"🗑️ Deleting {name} in {inst_zone}")
    return subprocess.call(cmd, shell=True)


def delete_all_tsm_instances(project):
    instances = fetch_existing_instances(project)

    print("\n➡ Existing instances to delete:")
    for name, inst_zone in instances:
        print(f"  - {name} ({inst_zone})")

    if instances:
        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances,
            max_workers=min(32, len(instances)),
        )
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


def prune_tsm_instances_for_next_run(project, next_num_nodes, n_clients):
    """
    Keep only instances needed for the next system size.

    For next_num_nodes = N and n_clients = 1, we keep:
        replicas: tsm-sc-000 ... tsm-sc-(N-1)
        client:   tsm-sc-N

    Everything with index > N is deleted.
    """
    keep_indices = set(range(next_num_nodes + n_clients))
    instances = fetch_existing_instances(project)

    instances_to_delete = []

    for name, inst_zone in instances:
        idx = parse_tsm_index(name)
        if idx not in keep_indices:
            instances_to_delete.append((name, inst_zone))

    print(
        "\n➡ Pruning instances for next run: "
        f"next_num_nodes={next_num_nodes}, keeping indices "
        f"0 through {next_num_nodes + n_clients - 1}"
    )

    if instances_to_delete:
        print("Instances to delete:")
        for name, inst_zone in instances_to_delete:
            print(f"  - {name} ({inst_zone})")

        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances_to_delete,
            max_workers=min(32, len(instances_to_delete)),
        )

        print("\n🧹 Non-needed instances deleted.\n")
    else:
        print("\n✔ No non-needed instances to delete.\n")


# ============================================================
# Experiment config
# ============================================================

# latencies: 50, 90, 150, 210
default_region = ["us-west1-b"]
# regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
regions = ["us-west1-b", "us-west1-b", "us-west1-b", "us-west1-b"]

zone_no = 0

project = "research-488322"
zone = "us-west1-b"
machine_type = "e2-standard-2"
image_family = "tsm-sc-family"
subnet = "default"
gcp_username = "tejas"

# Vary system size.
# Descending order is important because we can create 32 once,
# compile once, and then delete unnecessary higher-index nodes.
NUM_NODES_LIST = [4]

# Use 1 extra 2-core machine as client machine.
# For num_nodes = N, client is tsm-sc-N.
n_clients = 1

# Fixed collection attempts.
MAX_COLLECT_ATTEMPTS = 100

# Source file to patch before the first subrun.
OVERLAY_MANAGER_IMPL_PATH = Path("src/overlay/OverlayManagerImpl.cpp")

# Start clean only once, before the first run.
DELETE_BEFORE_FIRST_RUN = False

# After each subrun, delete nodes not needed for the next smaller run.
PRUNE_UNUSED_AFTER_EACH_RUN = True

# Delete remaining VMs after the final run.
DELETE_ALL_AFTER_FINAL_RUN = True

# Code does not change across system sizes now, so compile only first subrun.
COMPILE_ONLY_FIRST_RUN = True

MAX_NUM_NODES = max(NUM_NODES_LIST)

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed offered load.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]


# ============================================================
# Local source patching
# ============================================================

def patch_max_collect_attempts(max_collect_attempts):
    text = OVERLAY_MANAGER_IMPL_PATH.read_text()

    pattern = r"static\s+constexpr\s+uint64_t\s+MAX_COLLECT_ATTEMPTS\s*=\s*\d+\s*;"
    replacement = (
        f"static constexpr uint64_t MAX_COLLECT_ATTEMPTS = {max_collect_attempts};"
    )

    new_text, count = re.subn(
        pattern,
        replacement,
        text,
        flags=re.MULTILINE,
    )

    if count != 1:
        raise RuntimeError(
            f"Expected to replace exactly one MAX_COLLECT_ATTEMPTS line in "
            f"{OVERLAY_MANAGER_IMPL_PATH}, but replaced {count}"
        )

    OVERLAY_MANAGER_IMPL_PATH.write_text(new_text)

    print(
        f"Updated {OVERLAY_MANAGER_IMPL_PATH}: "
        f"MAX_COLLECT_ATTEMPTS = {max_collect_attempts}"
    )


def push_collect_patch_to_git(max_collect_attempts):
    cmd = (
        f'git add {OVERLAY_MANAGER_IMPL_PATH}; '
        f'git commit -m "set collect attempts {max_collect_attempts}" || true; '
        f'git push'
    )

    return subprocess.call(cmd, shell=True)


# ============================================================
# Main experiment loop
# ============================================================

for run_idx, num_nodes in enumerate(NUM_NODES_LIST):

    print("\n" + "#" * 100)
    print(
        f"Starting experiment for num_nodes={num_nodes}, "
        f"MAX_COLLECT_ATTEMPTS={MAX_COLLECT_ATTEMPTS}"
    )
    print("#" * 100)

    def get_zone_for_instance(i):
        # Use MAX_NUM_NODES so zone assignment stays stable.
        if i < int(MAX_NUM_NODES / 2):
            return default_region[0]
        else:
            return regions[zone_no]

    # -----------------------------------------------------------------
    # Delete existing tsm-sc-* instances only before the first subrun.
    # -----------------------------------------------------------------
    if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
        print("\n🧹 Cleaning before first run...")
        delete_all_tsm_instances(project)

    # -----------------------------------------------------------------
    # Create only missing replica/client machines for this system size.
    # -----------------------------------------------------------------
    existing_instances = fetch_existing_instances(project)
    existing_names = {name for name, _ in existing_instances}

    commands = []
    newly_created_indices = []

    # Create replica nodes only if they do not already exist.
    for i in range(num_nodes):
        instance_name = f"tsm-sc-{i:03}"

        if instance_name in existing_names:
            print(f"✔ Reusing existing replica {instance_name}")
            continue

        inst_zone = get_zone_for_instance(i)

        cmd = f'''
        gcloud compute instances create {instance_name} \
            --project={project} \
            --zone={inst_zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
        newly_created_indices.append(i)

    # Create client machine only if it does not already exist.
    for i in range(n_clients):
        client_idx = num_nodes + i
        instance_name = f"tsm-sc-{client_idx:03}"

        if instance_name in existing_names:
            print(f"✔ Reusing existing client {instance_name}")
            continue

        inst_zone = get_zone_for_instance(client_idx)

        cmd = f'''
        gcloud compute instances create {instance_name} \
            --project={project} \
            --zone={inst_zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
        newly_created_indices.append(client_idx)

    if commands:
        run_parallel(
            run_command,
            commands,
            max_workers=min(48, len(commands)),
        )

        print("All missing instances launched.")

        # Give GCP/SSH a little time after VM creation.
        time.sleep(30)
    else:
        print("✔ All required instances already exist; no VM creation needed.")

    # -----------------------------------------------------------------
    # Get sorted node and client IPs.
    # tsm_ips.txt must include only replica IPs, not clients.
    # -----------------------------------------------------------------
    ip_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --sort-by=name \
        --format="value(name,zone,networkInterfaces[0].networkIP)"
    '''

    ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

    instance_records = []

    for line in ip_output.splitlines():
        if line.strip():
            name, inst_zone, ip = line.split()
            idx = int(name.rsplit("-", 1)[1])
            instance_records.append((idx, name, inst_zone, ip))

    instance_records.sort()

    node_records = [r for r in instance_records if r[0] < num_nodes]
    client_records = [
        r for r in instance_records
        if num_nodes <= r[0] < num_nodes + n_clients
    ]

    if len(node_records) != num_nodes:
        raise RuntimeError(
            f"Expected {num_nodes} replica nodes, but found "
            f"{len(node_records)}: {node_records}"
        )

    if len(client_records) != n_clients:
        raise RuntimeError(
            f"Expected {n_clients} client nodes, but found "
            f"{len(client_records)}: {client_records}"
        )

    iplist = [r[3] for r in node_records]

    with open("tsm_ips.txt", "w") as f:
        for ip in iplist:
            f.write(ip + "\n")

    print("🎯 Node IPs:", iplist)
    print("🎯 Client instances:", client_records)

    node1_ip = iplist[0]
    print(f"Clients will connect to leader/node1 at: {node1_ip}")

    # -----------------------------------------------------------------
    # Remote helper functions.
    # -----------------------------------------------------------------

    def kill_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        remote_command = """\
sudo pkill -9 stellar-core || true; \
sudo pkill -9 shab_client || true; \
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")
        output = subprocess.call(command, shell=True)
        print(f"Return code for tsm-sc-{i:03}: {output}")
        return output

    def git_pull_stellar(i):
        inst_zone = get_zone_for_instance(i)

        command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

        print(command)
        output = subprocess.call(command, shell=True)
        print(output)
        return output

    def compile_stellar(i):
        inst_zone = get_zone_for_instance(i)

        command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j4; \
cd; \
sudo rm -rf stellar-private"'''

        print(command)
        output = subprocess.call(command, shell=True)
        print(output)
        return output

    def clean_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        remote_command = """\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")
        output = subprocess.call(command, shell=True)
        print(f"Return code for tsm-sc-{i:03}: {output}")
        return output

    def copy_folder_to_instance(
        i,
        source_folder="/home/tejas/stellar-private",
        destination_path="/home/tejas/stellar-private",
    ):
        inst_zone = get_zone_for_instance(i)
        instance_name = f"tsm-sc-{i:03}"

        command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

        print(f"Executing command for {instance_name}: {command}")

        output = subprocess.call(command, shell=True)

        print(f"Command for {instance_name} finished with exit code: {output}")

        return (instance_name, output)

    def run_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        node_number = i + 1
        instance_name = f"tsm-sc-{i:03}"

        remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")

        output = subprocess.call(command, shell=True)

        print(f"Return code for {instance_name}: {output}")
        return output

    def run_stellar_client(i, client_max_in_flight, client_threads):
        inst_zone = get_zone_for_instance(i)

        instance_name = f"tsm-sc-{i:03}"
        client_id = i - num_nodes

        remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing client command: {command}")

        output = subprocess.call(command, shell=True)

        print(f"Return code for {instance_name}: {output}")
        return output

    # -----------------------------------------------------------------
    # Generate stellar-private configs locally using only replica IPs.
    # -----------------------------------------------------------------
    stellar_private_path = Path("../stellar-private")
    if stellar_private_path.exists():
        shutil.rmtree(stellar_private_path)
    stellar_private_path.mkdir()

    subprocess.call(
        "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
        shell=True,
    )

    subprocess.call(
        "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
        "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
        shell=True,
    )

    # Enable custom message only on leader.
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg"

    subprocess.call(
        f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
        shell=True,
    )

    print("\n" + "*" * 100)
    print(
        f"Preparing subrun with num_nodes={num_nodes}, "
        f"MAX_COLLECT_ATTEMPTS={MAX_COLLECT_ATTEMPTS}"
    )
    print("*" * 100)

    # -----------------------------------------------------------------
    # Patch/push/compile only on the first subrun.
    #
    # Because MAX_COLLECT_ATTEMPTS is fixed at 50 and system sizes are
    # run in descending order, all later nodes are reused from the first
    # 32-node deployment and already have compiled code.
    # -----------------------------------------------------------------
    if run_idx == 0 or not COMPILE_ONLY_FIRST_RUN:
        # Patch local source.
        patch_max_collect_attempts(MAX_COLLECT_ATTEMPTS)

        # Push patched source.
        push_collect_patch_to_git(MAX_COLLECT_ATTEMPTS)

        # Stop any old processes.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Pull patched code on active VMs.
        results = run_parallel(
            git_pull_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Compile patched code on active VMs.
        results = run_parallel(
            compile_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)
    else:
        print(
            "\n✔ Skipping git pull and compile because "
            "COMPILE_ONLY_FIRST_RUN=True and this is not the first subrun."
        )

        # Still kill any stale processes on reused machines.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

    # -----------------------------------------------------------------
    # Throughput/latency loop.
    # -----------------------------------------------------------------
    for load in LOAD_POINTS:
        active_clients = load["active_clients"]
        client_threads = load["client_threads"]
        client_max_in_flight = load["max_in_flight"]

        if active_clients > n_clients:
            raise RuntimeError(
                f"active_clients={active_clients} exceeds n_clients={n_clients}"
            )

        total_client_threads = active_clients * client_threads
        aggregate_max_in_flight = (
            active_clients * client_threads * client_max_in_flight
        )

        run_label = (
            f"clients_{active_clients}_threads_{client_threads}_"
            f"inflight_{client_max_in_flight}_"
            f"total_threads_{total_client_threads}_"
            f"total_inflight_{aggregate_max_in_flight}"
        )

        print("\n" + "=" * 80)
        print(
            f"🚀 Starting run: num_nodes={num_nodes}, "
            f"MAX_COLLECT_ATTEMPTS={MAX_COLLECT_ATTEMPTS}, {run_label}"
        )
        print("=" * 80)

        # Clean remote stellar-private.
        results = run_parallel(
            clean_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Copy fresh stellar-private config to all active machines.
        results = run_parallel(
            copy_folder_to_instance,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Kill old processes on all active machines.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Start consensus replicas.
        results = run_parallel(
            run_stellar_private,
            range(num_nodes),
            max_workers=min(48, num_nodes),
        )

        print(results)
        print("All Stellar nodes should be starting in the background.")

        # Give nodes time to authenticate and start the client listener.
        time.sleep(60)

        # Start only required client VMs for this load point.
        active_client_indices = [
            num_nodes + j for j in range(active_clients)
        ]

        results = run_parallel(
            lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
            active_client_indices,
            max_workers=active_clients,
        )

        print(results)
        print(
            f"Started {active_clients} client VM(s), "
            f"each with {client_threads} client threads. "
            f"Total client threads = {total_client_threads}. "
            f"Aggregate max in-flight = {aggregate_max_in_flight}."
        )

        # Let the system run.
        time.sleep(CLIENT_WAIT_AFTER_START_SEC)

        # Stop all nodes and clients.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # -------------------------------------------------------------
        # Save logs.
        # -------------------------------------------------------------
        remote_base_folder = "/home/tejas/stellar-private"

        local_base_destination = (
            "/home/tejas/work/experiments/shabdiz/"
            + f"CollectionVSNumNodes_{MAX_COLLECT_ATTEMPTS}_"
            + f"nodes_{num_nodes}_{run_label}"
        )

        Path(local_base_destination).mkdir(parents=True, exist_ok=True)

        # Save run metadata.
        with open(Path(local_base_destination) / "run_config.txt", "w") as f:
            f.write(f"num_nodes={num_nodes}\n")
            f.write(f"max_collect_attempts={MAX_COLLECT_ATTEMPTS}\n")
            f.write(f"force_collect_after_sec=100\n")
            f.write(f"active_clients={active_clients}\n")
            f.write(f"client_threads_per_vm={client_threads}\n")
            f.write(f"total_client_threads={total_client_threads}\n")
            f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
            f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
            f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
            f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
            f.write(f"leader_ip={node1_ip}\n")
            f.write(f"machine_type={machine_type}\n")
            f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

        def copy_folder_from_instance(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"

            node_number = i + 1
            node_folder = f"node{node_number}"

            remote_source_path = posixpath.join(remote_base_folder, node_folder)

            local_destination_path = Path(local_base_destination) / instance_name
            local_destination_path.mkdir(parents=True, exist_ok=True)

            remote_source = f"{instance_name}:{remote_source_path}"

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

            print(
                f"Executing command to copy {node_folder} "
                f"from {instance_name}: {command}"
            )

            output = subprocess.call(command, shell=True)

            print(f"Copy from {instance_name} finished with exit code: {output}")

            return (instance_name, output)

        # Copy only first 3 replica logs.
        node_copy_results = run_parallel(
            copy_folder_from_instance,
            range(3),
            max_workers=3,
        )

        def copy_client_log(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"
            client_id = i - num_nodes

            remote_source = (
                f"{instance_name}:/home/tejas/stellar-private/"
                f"stellar-client-{client_id}.log"
            )

            local_destination_path = Path(local_base_destination)
            local_destination_path.mkdir(parents=True, exist_ok=True)

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

            print(f"Copying client log from {instance_name}...")

            output = subprocess.call(command, shell=True)

            print(f"Copy finished with exit code: {output}")

            return (instance_name, output)

        client_copy_results = run_parallel(
            copy_client_log,
            active_client_indices,
            max_workers=active_clients,
        )

        print("\n--- Summary of Download Results ---")
        print("Node log copies:", node_copy_results)
        print("Client log copies:", client_copy_results)
        print(f"Saved run to: {local_base_destination}")

    # -----------------------------------------------------------------
    # After each system-size subrun, delete nodes not needed for the next.
    # -----------------------------------------------------------------
    is_last_run = run_idx == len(NUM_NODES_LIST) - 1

    if PRUNE_UNUSED_AFTER_EACH_RUN and not is_last_run:
        next_num_nodes = NUM_NODES_LIST[run_idx + 1]
        prune_tsm_instances_for_next_run(
            project=project,
            next_num_nodes=next_num_nodes,
            n_clients=n_clients,
        )

    elif is_last_run and DELETE_ALL_AFTER_FINAL_RUN:
        print("\n🧹 Final run completed. Deleting all remaining VMs...")
        delete_all_tsm_instances(project)

# Collection vs Network Latency

In [ ]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
import re
from pathlib import Path

import os
# ============================================================
# Helpers
# ============================================================

def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


def fetch_existing_instances(project):
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''

    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []

    for line in output.splitlines():
        if line.strip():
            name, inst_zone = line.split()
            instances.append((name, inst_zone))

    return instances


def delete_instance(project, instance):
    name, inst_zone = instance

    cmd = f'''
    gcloud compute instances delete {name} \
        --zone={inst_zone} \
        --project={project} \
        --quiet
    '''

    print(f"🗑️ Deleting {name} in {inst_zone}")
    return subprocess.call(cmd, shell=True)


def delete_all_tsm_instances(project):
    instances = fetch_existing_instances(project)

    print("\n➡ Existing instances to delete:")
    for name, inst_zone in instances:
        print(f"  - {name} ({inst_zone})")

    if instances:
        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances,
            max_workers=min(32, len(instances)),
        )
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


# ============================================================
# Experiment config
# ============================================================

project = "research-488322"
machine_type = "e2-standard-2"
image_family = "tsm-sc-family"
subnet = "default"
gcp_username = "tejas"

# Fixed system size.
NUM_NODES = 16

# One client VM.
# Client index will be tsm-sc-016.
n_clients = 1

# Fixed collection attempts.
COLLECT_ATTEMPT_POINTS = [5]

# First 8 replicas are always here.
BASE_REPLICA_ZONE = "us-west1-b"

# Client is always here.
CLIENT_ZONE = "us-west1-b"

# Replicas 8..15 sweep over these remote zones.
REMOTE_REPLICA_ZONES = [
    # "asia-northeast1-b",
    "europe-west3-c",
    # "asia-south1-c",
    "northamerica-northeast2-b",
    # "us-west1-b"
]

# Source file to patch before each subrun.
OVERLAY_MANAGER_IMPL_PATH = Path("src/overlay/OverlayManagerImpl.cpp")

# Local stellar-private path.
LOCAL_STELLAR_PRIVATE_PATH = Path("../stellar-private").resolve()

# Delete before first run.
DELETE_BEFORE_FIRST_RUN = True

# Delete all VMs after every subrun.
DELETE_AFTER_EACH_SUBRUN = True

# Since VMs are deleted after every subrun, compile every subrun.
COMPILE_EVERY_SUBRUN = True

# Copy only these replica logs.
# Change to [0, 8, 15] if you want one local and two remote-node logs.
COPY_NODE_INDICES = [0, 1, 2]

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed offered load.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]


# ============================================================
# Local source patching
# ============================================================

def patch_max_collect_attempts(max_collect_attempts):
    text = OVERLAY_MANAGER_IMPL_PATH.read_text()

    pattern = r"static\s+constexpr\s+uint64_t\s+MAX_COLLECT_ATTEMPTS\s*=\s*\d+\s*;"
    replacement = (
        f"static constexpr uint64_t MAX_COLLECT_ATTEMPTS = {max_collect_attempts};"
    )

    new_text, count = re.subn(
        pattern,
        replacement,
        text,
        flags=re.MULTILINE,
    )

    if count != 1:
        raise RuntimeError(
            f"Expected to replace exactly one MAX_COLLECT_ATTEMPTS line in "
            f"{OVERLAY_MANAGER_IMPL_PATH}, but replaced {count}"
        )

    OVERLAY_MANAGER_IMPL_PATH.write_text(new_text)

    print(
        f"Updated {OVERLAY_MANAGER_IMPL_PATH}: "
        f"MAX_COLLECT_ATTEMPTS = {max_collect_attempts}"
    )


def push_collect_patch_to_git(max_collect_attempts):
    cmd = (
        f'git add .; '
        f'git commit -m "set collect attempts {max_collect_attempts}" || true; '
        f'git push'
    )

    return subprocess.call(cmd, shell=True)


# ============================================================
# Zone assignment
# ============================================================

def get_zone_for_instance(i, remote_replica_zone):
    """
    Replica layout for NUM_NODES = 16:
        tsm-sc-000 ... tsm-sc-007 -> us-west1-b
        tsm-sc-008 ... tsm-sc-015 -> remote_replica_zone
        tsm-sc-016                -> us-west1-b client
    """
    if i < 8:
        return BASE_REPLICA_ZONE

    if i < NUM_NODES:
        return remote_replica_zone

    return CLIENT_ZONE


# ============================================================
# GCP instance creation
# ============================================================

def create_instance_command(instance_name, inst_zone):
    return f'''
    gcloud compute instances create {instance_name} \
        --project={project} \
        --zone={inst_zone} \
        --machine-type={machine_type} \
        --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
        --can-ip-forward \
        --maintenance-policy=MIGRATE \
        --provisioning-model=STANDARD \
        --service-account=254510644191-compute@developer.gserviceaccount.com \
        --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
        --tags=http-server,https-server \
        --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
        --no-shielded-secure-boot \
        --shielded-vtpm \
        --shielded-integrity-monitoring \
        --labels=goog-ec-src=vm_add-gcloud \
        --reservation-affinity=any
    '''.strip()


# ============================================================
# Main experiment loop
# ============================================================
os.system('git add .; git commit -m "update"; git push')
subrun_idx = 0

for remote_replica_zone in REMOTE_REPLICA_ZONES:
    for max_collect_attempts in COLLECT_ATTEMPT_POINTS:

        subrun_idx += 1

        print("\n" + "#" * 100)
        print(
            f"Starting network-latency subrun {subrun_idx}: "
            f"num_nodes={NUM_NODES}, "
            f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, "
            f"first_8_zone={BASE_REPLICA_ZONE}, "
            f"next_8_zone={remote_replica_zone}, "
            f"client_zone={CLIENT_ZONE}"
        )
        print("#" * 100)

        if DELETE_BEFORE_FIRST_RUN and subrun_idx == 1:
            print("\n🧹 Cleaning before first run...")
            delete_all_tsm_instances(project)

        try:
            # ------------------------------------------------------------
            # Create all replica/client machines for this subrun.
            # Since we delete after every subrun, these are usually fresh.
            # ------------------------------------------------------------
            existing_instances = fetch_existing_instances(project)
            existing_names = {name for name, _ in existing_instances}

            commands = []

            # Create replica nodes.
            for i in range(NUM_NODES):
                instance_name = f"tsm-sc-{i:03}"

                if instance_name in existing_names:
                    print(f"✔ Reusing existing replica {instance_name}")
                    continue

                inst_zone = get_zone_for_instance(i, remote_replica_zone)
                commands.append(create_instance_command(instance_name, inst_zone))

            # Create client node.
            for i in range(n_clients):
                client_idx = NUM_NODES + i
                instance_name = f"tsm-sc-{client_idx:03}"

                if instance_name in existing_names:
                    print(f"✔ Reusing existing client {instance_name}")
                    continue

                inst_zone = get_zone_for_instance(client_idx, remote_replica_zone)
                commands.append(create_instance_command(instance_name, inst_zone))

            if commands:
                run_parallel(
                    run_command,
                    commands,
                    max_workers=min(48, len(commands)),
                )

                print("All missing instances launched.")
                time.sleep(30)
            else:
                print("✔ All required instances already exist; no VM creation needed.")

            # ------------------------------------------------------------
            # Get sorted replica and client IPs.
            # tsm_ips.txt must include only replica IPs, not clients.
            # ------------------------------------------------------------
            ip_cmd = f'''
            gcloud compute instances list \
                --project={project} \
                --filter="name~'^tsm-sc-'" \
                --sort-by=name \
                --format="value(name,zone,networkInterfaces[0].networkIP)"
            '''

            ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

            instance_records = []

            for line in ip_output.splitlines():
                if line.strip():
                    name, inst_zone, ip = line.split()
                    idx = int(name.rsplit("-", 1)[1])
                    instance_records.append((idx, name, inst_zone, ip))

            instance_records.sort()

            node_records = [r for r in instance_records if r[0] < NUM_NODES]
            client_records = [
                r for r in instance_records
                if NUM_NODES <= r[0] < NUM_NODES + n_clients
            ]

            if len(node_records) != NUM_NODES:
                raise RuntimeError(
                    f"Expected {NUM_NODES} replica nodes, but found "
                    f"{len(node_records)}: {node_records}"
                )

            if len(client_records) != n_clients:
                raise RuntimeError(
                    f"Expected {n_clients} client nodes, but found "
                    f"{len(client_records)}: {client_records}"
                )

            iplist = [r[3] for r in node_records]

            with open("tsm_ips.txt", "w") as f:
                for ip in iplist:
                    f.write(ip + "\n")

            print("🎯 Node IPs:", iplist)
            print("🎯 Client instances:", client_records)

            node1_ip = iplist[0]
            print(f"Clients will connect to leader/node1 at: {node1_ip}")

            # ------------------------------------------------------------
            # Remote helper functions.
            # ------------------------------------------------------------

            def kill_stellar_private(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                remote_command = """\
sudo pkill -9 stellar-core || true; \
sudo pkill -9 shab_client || true; \
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing: {command}")
                output = subprocess.call(command, shell=True)
                print(f"Return code for tsm-sc-{i:03}: {output}")
                return output

            def git_pull_stellar(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

                print(command)
                output = subprocess.call(command, shell=True)
                print(output)
                return output

            def compile_stellar(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j4; \
cd; \
sudo rm -rf stellar-private"'''

                print(command)
                output = subprocess.call(command, shell=True)
                print(output)
                return output

            def clean_stellar_private(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                remote_command = """\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing: {command}")
                output = subprocess.call(command, shell=True)
                print(f"Return code for tsm-sc-{i:03}: {output}")
                return output

            def copy_folder_to_instance(
                i,
                source_folder=None,
                destination_path="/home/tejas/stellar-private",
            ):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)
                instance_name = f"tsm-sc-{i:03}"

                if source_folder is None:
                    source_folder = str(LOCAL_STELLAR_PRIVATE_PATH)

                command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

                print(f"Executing command for {instance_name}: {command}")

                output = subprocess.call(command, shell=True)

                print(f"Command for {instance_name} finished with exit code: {output}")

                return (instance_name, output)

            def run_stellar_private(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                node_number = i + 1
                instance_name = f"tsm-sc-{i:03}"

                remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing: {command}")

                output = subprocess.call(command, shell=True)

                print(f"Return code for {instance_name}: {output}")
                return output

            def run_stellar_client(i, client_max_in_flight, client_threads):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                instance_name = f"tsm-sc-{i:03}"
                client_id = i - NUM_NODES

                remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing client command: {command}")

                output = subprocess.call(command, shell=True)

                print(f"Return code for {instance_name}: {output}")
                return output

            # ------------------------------------------------------------
            # Generate stellar-private configs locally using only replica IPs.
            # This must run every subrun because IPs/zones change.
            # ------------------------------------------------------------
            if LOCAL_STELLAR_PRIVATE_PATH.exists():
                shutil.rmtree(LOCAL_STELLAR_PRIVATE_PATH)
            LOCAL_STELLAR_PRIVATE_PATH.mkdir(parents=True, exist_ok=True)

            subprocess.call(
                f"cp gcp_setup_stellar_private.sh "
                f"{LOCAL_STELLAR_PRIVATE_PATH}/gcp_setup_stellar_private.sh",
                shell=True,
            )

            subprocess.call(
                f"cd {LOCAL_STELLAR_PRIVATE_PATH}; "
                f"chmod +x gcp_setup_stellar_private.sh; "
                f"./gcp_setup_stellar_private.sh start; "
                f"./gcp_setup_stellar_private.sh",
                shell=True,
            )

            # Enable custom message only on leader.
            line_to_add = "SEND_CUSTOM_MESSAGE=true"
            target_file = LOCAL_STELLAR_PRIVATE_PATH / "node1" / "stellar-core.cfg"

            subprocess.call(
                f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
                shell=True,
            )

            print("\n" + "*" * 100)
            print(
                f"Preparing subrun with "
                f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, "
                f"remote_replica_zone={remote_replica_zone}"
            )
            print("*" * 100)

            # ------------------------------------------------------------
            # Patch, push, pull, and compile every subrun.
            # ------------------------------------------------------------
            patch_max_collect_attempts(max_collect_attempts)
            push_collect_patch_to_git(max_collect_attempts)

            results = run_parallel(
                kill_stellar_private,
                range(NUM_NODES + n_clients),
                max_workers=min(48, NUM_NODES + n_clients),
            )
            print(results)

            results = run_parallel(
                git_pull_stellar,
                range(NUM_NODES + n_clients),
                max_workers=min(48, NUM_NODES + n_clients),
            )
            print(results)

            results = run_parallel(
                compile_stellar,
                range(NUM_NODES + n_clients),
                max_workers=min(48, NUM_NODES + n_clients),
            )
            print(results)

            # ------------------------------------------------------------
            # Throughput/latency loop.
            # ------------------------------------------------------------
            for load in LOAD_POINTS:
                active_clients = load["active_clients"]
                client_threads = load["client_threads"]
                client_max_in_flight = load["max_in_flight"]

                if active_clients > n_clients:
                    raise RuntimeError(
                        f"active_clients={active_clients} exceeds n_clients={n_clients}"
                    )

                total_client_threads = active_clients * client_threads
                aggregate_max_in_flight = (
                    active_clients * client_threads * client_max_in_flight
                )

                run_label = (
                    f"clients_{active_clients}_threads_{client_threads}_"
                    f"inflight_{client_max_in_flight}_"
                    f"total_threads_{total_client_threads}_"
                    f"total_inflight_{aggregate_max_in_flight}"
                )

                print("\n" + "=" * 80)
                print(
                    f"🚀 Starting run: "
                    f"num_nodes={NUM_NODES}, "
                    f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, "
                    f"remote_replica_zone={remote_replica_zone}, "
                    f"{run_label}"
                )
                print("=" * 80)

                # Clean remote stellar-private.
                results = run_parallel(
                    clean_stellar_private,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # Copy fresh stellar-private config to all active machines.
                results = run_parallel(
                    copy_folder_to_instance,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # Kill old processes on all active machines.
                results = run_parallel(
                    kill_stellar_private,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # Start consensus replicas.
                results = run_parallel(
                    run_stellar_private,
                    range(NUM_NODES),
                    max_workers=min(48, NUM_NODES),
                )

                print(results)
                print("All Stellar nodes should be starting in the background.")

                # Give nodes time to authenticate and start the client listener.
                time.sleep(40)

                # Start only required client VMs for this load point.
                active_client_indices = [
                    NUM_NODES + j for j in range(active_clients)
                ]

                results = run_parallel(
                    lambda i: run_stellar_client(
                        i,
                        client_max_in_flight,
                        client_threads,
                    ),
                    active_client_indices,
                    max_workers=active_clients,
                )

                print(results)
                print(
                    f"Started {active_clients} client VM(s), "
                    f"each with {client_threads} client threads. "
                    f"Total client threads = {total_client_threads}. "
                    f"Aggregate max in-flight = {aggregate_max_in_flight}."
                )

                # Let the system run.
                time.sleep(CLIENT_WAIT_AFTER_START_SEC)

                # Stop all nodes and clients.
                results = run_parallel(
                    kill_stellar_private,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # --------------------------------------------------------
                # Save logs.
                # --------------------------------------------------------
                remote_base_folder = "/home/tejas/stellar-private"

                local_base_destination = (
                    "/home/tejas/work/experiments/shabdiz/"
                    + f"CollectionVSNetworkLatency_{max_collect_attempts}_"
                    + f"remote_{remote_replica_zone}_"
                    + f"nodes_{NUM_NODES}_{run_label}"
                )

                Path(local_base_destination).mkdir(parents=True, exist_ok=True)

                with open(Path(local_base_destination) / "run_config.txt", "w") as f:
                    f.write(f"experiment=CollectionVSNetworkLatency\n")
                    f.write(f"num_nodes={NUM_NODES}\n")
                    f.write(f"max_collect_attempts={max_collect_attempts}\n")
                    f.write(f"force_collect_after_sec=100\n")
                    f.write(f"base_replica_zone={BASE_REPLICA_ZONE}\n")
                    f.write(f"remote_replica_zone={remote_replica_zone}\n")
                    f.write(f"client_zone={CLIENT_ZONE}\n")
                    f.write(f"replicas_0_to_7_zone={BASE_REPLICA_ZONE}\n")
                    f.write(f"replicas_8_to_15_zone={remote_replica_zone}\n")
                    f.write(f"client_instance=tsm-sc-{NUM_NODES:03}\n")
                    f.write(f"active_clients={active_clients}\n")
                    f.write(f"client_threads_per_vm={client_threads}\n")
                    f.write(f"total_client_threads={total_client_threads}\n")
                    f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
                    f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
                    f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
                    f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
                    f.write(f"leader_ip={node1_ip}\n")
                    f.write(f"machine_type={machine_type}\n")
                    f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

                def copy_folder_from_instance(i):
                    inst_zone = get_zone_for_instance(i, remote_replica_zone)

                    instance_name = f"tsm-sc-{i:03}"

                    node_number = i + 1
                    node_folder = f"node{node_number}"

                    remote_source_path = posixpath.join(remote_base_folder, node_folder)

                    local_destination_path = Path(local_base_destination) / instance_name
                    local_destination_path.mkdir(parents=True, exist_ok=True)

                    remote_source = f"{instance_name}:{remote_source_path}"

                    command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

                    print(
                        f"Executing command to copy {node_folder} "
                        f"from {instance_name}: {command}"
                    )

                    output = subprocess.call(command, shell=True)

                    print(f"Copy from {instance_name} finished with exit code: {output}")

                    return (instance_name, output)

                node_copy_results = run_parallel(
                    copy_folder_from_instance,
                    COPY_NODE_INDICES,
                    max_workers=min(3, len(COPY_NODE_INDICES)),
                )

                def copy_client_log(i):
                    inst_zone = get_zone_for_instance(i, remote_replica_zone)

                    instance_name = f"tsm-sc-{i:03}"
                    client_id = i - NUM_NODES

                    remote_source = (
                        f"{instance_name}:/home/tejas/stellar-private/"
                        f"stellar-client-{client_id}.log"
                    )

                    local_destination_path = Path(local_base_destination)
                    local_destination_path.mkdir(parents=True, exist_ok=True)

                    command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

                    print(f"Copying client log from {instance_name}...")

                    output = subprocess.call(command, shell=True)

                    print(f"Copy finished with exit code: {output}")

                    return (instance_name, output)

                client_copy_results = run_parallel(
                    copy_client_log,
                    active_client_indices,
                    max_workers=active_clients,
                )

                print("\n--- Summary of Download Results ---")
                print("Node log copies:", node_copy_results)
                print("Client log copies:", client_copy_results)
                print(f"Saved run to: {local_base_destination}")

        finally:
            if DELETE_AFTER_EACH_SUBRUN:
                print(
                    "\n🧹 Deleting VMs after completed/attempted subrun: "
                    f"remote_replica_zone={remote_replica_zone}, "
                    f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}"
                )
                delete_all_tsm_instances(project)

In [ ]:

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")

In [ ]:
#             # Clean remote stellar-private.
#             results = run_parallel(
#                 clean_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Copy fresh stellar-private config to all active machines.
#             results = run_parallel(
#                 copy_folder_to_instance,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Kill old processes on all active machines.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Start consensus replicas.
#             results = run_parallel(
#                 run_stellar_private,
#                 range(num_nodes),
#                 max_workers=min(48, num_nodes),
#             )

#             print(results)
#             print("All Stellar nodes should be starting in the background.")

#             # Give nodes time to authenticate and start the client listener.
#             time.sleep(60)

#             # Start only required client VMs for this load point.
#             active_client_indices = [
#                 num_nodes + j for j in range(active_clients)
#             ]

#             results = run_parallel(
#                 lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print(results)
#             print(
#                 f"Started {active_clients} client VM(s), "
#                 f"each with {client_threads} client threads. "
#                 f"Total client threads = {total_client_threads}. "
#                 f"Aggregate max in-flight = {aggregate_max_in_flight}."
#             )

#             # Let the system run before forcing failures.
#             time.sleep(CLIENT_WAIT_AFTER_START_SEC)


#             # Stop all nodes and clients.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # -----------------------------------------------------------------
#             # Save logs.
#             # -----------------------------------------------------------------
#             remote_base_folder = "/home/tejas/stellar-private"

#             local_base_destination = (
#                 "/home/tejas/work/experiments/shabdiz/"
#                 + f"Collection_{max_collect_attempts}_"
#                 + f"nodes_{num_nodes}_{run_label}"
#             )

#             Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#             # Save run metadata.
#             with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#                 f.write(f"num_nodes={num_nodes}\n")
#                 f.write(f"max_collect_attempts={max_collect_attempts}\n")
#                 f.write(f"force_collect_after_sec=100\n")
#                 f.write(f"active_clients={active_clients}\n")
#                 f.write(f"client_threads_per_vm={client_threads}\n")
#                 f.write(f"total_client_threads={total_client_threads}\n")
#                 f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#                 f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#                 f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#                 f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#                 f.write(f"leader_ip={node1_ip}\n")
#                 f.write(f"machine_type={machine_type}\n")
#                 f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#             def copy_folder_from_instance(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"

#                 node_number = i + 1
#                 node_folder = f"node{node_number}"

#                 remote_source_path = posixpath.join(remote_base_folder, node_folder)

#                 local_destination_path = Path(local_base_destination) / instance_name
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 remote_source = f"{instance_name}:{remote_source_path}"

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#                 print(
#                     f"Executing command to copy {node_folder} "
#                     f"from {instance_name}: {command}"
#                 )

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy from {instance_name} finished with exit code: {output}")

#                 return (instance_name, output)

#             # Copy all replica logs.
#             node_copy_results = run_parallel(
#                 copy_folder_from_instance,
#                 range(3),
#                 max_workers=min(48, max(1, num_nodes)),
#             )

#             def copy_client_log(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"
#                 client_id = i - num_nodes

#                 remote_source = (
#                     f"{instance_name}:/home/tejas/stellar-private/"
#                     f"stellar-client-{client_id}.log"
#                 )

#                 local_destination_path = Path(local_base_destination)
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#                 print(f"Copying client log from {instance_name}...")

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy finished with exit code: {output}")

#                 return (instance_name, output)

#             client_copy_results = run_parallel(
#                 copy_client_log,
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print("\n--- Summary of Download Results ---")
#             print("Node log copies:", node_copy_results)
#             print("Client log copies:", client_copy_results)
#             print(f"Saved run to: {local_base_destination}")

In [ ]:

# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# import re
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # =============================================================================
# # Fixed read-ratio experiment configuration
# # =============================================================================

# # Fixed system size for performance-vs-read-ratio experiment.
# NUM_NODES = 8

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = 8, client is tsm-sc-008.
# n_clients = 1

# # Read-ratio workloads. These must already exist in CustomYCSBWorkload.h:
# # WORKLOAD_R0, WORKLOAD_R25, WORKLOAD_R50, WORKLOAD_R75, WORKLOAD_R100
# READ_RATIO_WORKLOADS = [
#     # {"read_ratio": 0,   "workload": "WORKLOAD_R0"},
#     {"read_ratio": 10,   "workload": "WORKLOAD_R10"},
#     # {"read_ratio": 25,  "workload": "WORKLOAD_R25"},
#     # {"read_ratio": 50,  "workload": "WORKLOAD_R50"},
#     # {"read_ratio": 75,  "workload": "WORKLOAD_R75"},
#     # {"read_ratio": 20,  "workload": "WORKLOAD_R20"},
#     {"read_ratio": 40,  "workload": "WORKLOAD_R40"},
#     # {"read_ratio": 60,  "workload": "WORKLOAD_R60"},
#     # {"read_ratio": 80,  "workload": "WORKLOAD_R80"},
#     # {"read_ratio": 100, "workload": "WORKLOAD_R100"},
# ]

# # The file containing:
# #   static YCSBWorkload currentWorkload = WORKLOAD_R50;
# YCSB_HEADER_LOCAL = Path("src/overlay/CustomYCSBWorkload.h")

# # Start clean before the first read-ratio run.
# DELETE_BEFORE_FIRST_RUN = True

# # Delete all VMs after the final read-ratio run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # Compile stellar-core only in the first read-ratio subrun.
# # The client is compiled every subrun because CustomYCSBWorkload.h changes.
# COMPILE_STELLAR_CORE_ONLY_FIRST_RUN = True

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 210
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Same fixed client load as your previous experiment.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]

# # GCP settings.
# project = "research-488322"
# zone = "us-central1-c"
# machine_type = "e2-standard-2"
# image_family = "tsm-sc-family"  # your custom image
# subnet = "default"
# gcp_username = "tejas"

# # latencies: 50, 90 150, 210
# default_region = ["us-central1-a"]
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
# regions = ["us-central1-a", "us-central1-a", "us-central1-a", "us-central1-a"]

# zone_no = 0


# def get_zone_for_instance(i):
#     # Keep the same placement rule as your previous script.
#     if i < int(NUM_NODES / 2):
#         return default_region[0]
#     else:
#         return regions[zone_no]


# def set_local_ycsb_workload(workload_name):
#     """
#     Updates this line in src/overlay/CustomYCSBWorkload.h:

#         static YCSBWorkload currentWorkload = WORKLOAD_R50;

#     to the requested workload.
#     """
#     if not YCSB_HEADER_LOCAL.exists():
#         raise FileNotFoundError(
#             f"Could not find {YCSB_HEADER_LOCAL}. "
#             "Run this script from the stellar-core repo root."
#         )

#     text = YCSB_HEADER_LOCAL.read_text()

#     new_text, count = re.subn(
#         r"static\s+YCSBWorkload\s+currentWorkload\s*=\s*WORKLOAD_[A-Za-z0-9_]+\s*;",
#         f"static YCSBWorkload currentWorkload = {workload_name};",
#         text,
#         count=1,
#     )

#     if count != 1:
#         raise RuntimeError(
#             "Could not find exactly one currentWorkload line in "
#             f"{YCSB_HEADER_LOCAL}"
#         )

#     YCSB_HEADER_LOCAL.write_text(new_text)
#     print(f"✅ Updated {YCSB_HEADER_LOCAL}: currentWorkload = {workload_name}")


# def fetch_existing_instances():
#     fetch_cmd = f"""
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     """

#     output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     return instances


# def delete_instance(instance):
#     name, inst_zone = instance

#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={project} \
#         --quiet
#     """

#     print(f"🗑️ Deleting {name} in {inst_zone}")
#     return subprocess.call(cmd, shell=True)


# def parse_tsm_index(name):
#     return int(name.rsplit("-", 1)[1])


# def run_command(command):
#     print(f"Running: {command}")
#     return subprocess.call(command, shell=True)


# def kill_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     remote_command = f"""\
# cd /home/tejas/stellar-private 2>/dev/null || true; \
# sudo pkill -9 stellar-core || true; sudo pkill -9 shab_client || true; \
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Return code for tsm-sc-{i:03}: {output}")
#     return output


# def clean_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Return code for tsm-sc-{i:03}: {output}")
#     return output


# def copy_folder_to_instance(
#     i,
#     source_folder="/home/tejas/stellar-private",
#     destination_path="/home/tejas/stellar-private",
# ):
#     inst_zone = get_zone_for_instance(i)
#     instance_name = f"tsm-sc-{i:03}"

#     command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}" """

#     print(f"Executing command for {instance_name}: {command}")

#     output = subprocess.call(command, shell=True)

#     print(f"Command for {instance_name} finished with exit code: {output}")

#     return (instance_name, output)


# def run_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     node_number = i + 1
#     instance_name = f"tsm-sc-{i:03}"

#     remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")

#     output = subprocess.call(command, shell=True)

#     print(f"Return code for {instance_name}: {output}")
#     return output


# def git_pull_stellar(i):
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# def compile_stellar_and_client(i):
#     """
#     First subrun only.
#     This compiles both shab_client and stellar-core.
#     """
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# def copy_ycsb_header_to_instance(i):
#     """
#     Later subruns only.
#     Copy changed CustomYCSBWorkload.h to the VM source tree.
#     This keeps the source file in sync. We only recompile the client binary after this.
#     """
#     inst_zone = get_zone_for_instance(i)
#     instance_name = f"tsm-sc-{i:03}"

#     remote_path = (
#         f"{instance_name}:/home/tejas/stellar-core/src/overlay/"
#         f"{YCSB_HEADER_LOCAL.name}"
#     )

#     command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{YCSB_HEADER_LOCAL}" "{remote_path}" """

#     print(f"Copying workload header to {instance_name}: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Header copy to {instance_name} finished with exit code: {output}")
#     return (instance_name, output)


# def compile_client_only(i):
#     """
#     Every subrun after the first.
#     This recompiles only shab_client, not stellar-core.
#     """
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd /home/tejas/stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# # =============================================================================
# # VM setup
# # =============================================================================

# if DELETE_BEFORE_FIRST_RUN:
#     instances = fetch_existing_instances()

#     print("\n➡ Existing instances to delete before first run:")
#     for name, inst_zone in instances:
#         print(f"  - {name} ({inst_zone})")

#     if instances:
#         run_parallel(
#             delete_instance,
#             instances,
#             max_workers=min(32, len(instances)),
#         )
#         print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#     else:
#         print("\n✔ No existing tsm-sc-* instances found before first run.\n")


# # Create required replica/client machines.
# existing_instances = fetch_existing_instances()
# existing_names = {name for name, _ in existing_instances}

# commands = []

# # Replicas: tsm-sc-000 ... tsm-sc-007
# for i in range(NUM_NODES):
#     instance_name = f"tsm-sc-{i:03}"

#     if instance_name in existing_names:
#         print(f"✔ Reusing existing replica {instance_name}")
#         continue

#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances create {instance_name} \
#         --project={project} \
#         --zone={inst_zone} \
#         --machine-type={machine_type} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     commands.append(cmd.strip())

# # Client: tsm-sc-008
# for i in range(n_clients):
#     client_idx = NUM_NODES + i
#     instance_name = f"tsm-sc-{client_idx:03}"

#     if instance_name in existing_names:
#         print(f"✔ Reusing existing client {instance_name}")
#         continue

#     inst_zone = get_zone_for_instance(client_idx)

#     cmd = f"""
#     gcloud compute instances create {instance_name} \
#         --project={project} \
#         --zone={inst_zone} \
#         --machine-type={machine_type} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     commands.append(cmd.strip())

# if commands:
#     run_parallel(
#         run_command,
#         commands,
#         max_workers=min(48, len(commands)),
#     )

#     print("All missing instances launched.")

#     # Give GCP/SSH a little time after VM creation.
#     time.sleep(30)
# else:
#     print("✔ All required instances already exist; no VM creation needed.")


# # Get sorted node and client IPs.
# ip_cmd = f"""
# gcloud compute instances list \
#     --project={project} \
#     --filter="name~'^tsm-sc-'" \
#     --sort-by=name \
#     --format="value(name,zone,networkInterfaces[0].networkIP)"
# """

# ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

# instance_records = []

# for line in ip_output.splitlines():
#     if line.strip():
#         name, inst_zone, ip = line.split()
#         idx = int(name.rsplit("-", 1)[1])
#         instance_records.append((idx, name, inst_zone, ip))

# instance_records.sort()

# node_records = [r for r in instance_records if r[0] < NUM_NODES]
# client_records = [r for r in instance_records if NUM_NODES <= r[0] < NUM_NODES + n_clients]

# if len(node_records) != NUM_NODES:
#     raise RuntimeError(
#         f"Expected {NUM_NODES} replica nodes, but found {len(node_records)}: {node_records}"
#     )

# if len(client_records) != n_clients:
#     raise RuntimeError(
#         f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#     )

# iplist = [r[3] for r in node_records]

# with open("tsm_ips.txt", "w") as f:
#     for ip in iplist:
#         f.write(ip + "\n")

# print("🎯 Node IPs:", iplist)
# print("🎯 Client instances:", client_records)

# node1_ip = iplist[0]
# print(f"Clients will connect to leader/node1 at: {node1_ip}")

# all_instance_indices = list(range(NUM_NODES + n_clients))


# # =============================================================================
# # Read-ratio experiment loop
# # =============================================================================

# for workload_idx, workload_cfg in enumerate(READ_RATIO_WORKLOADS):
#     read_ratio = workload_cfg["read_ratio"]
#     workload_name = workload_cfg["workload"]

#     print("\n" + "#" * 100)
#     print(
#         f"Starting read-ratio experiment: "
#         f"num_nodes={NUM_NODES}, read_ratio={read_ratio}%, workload={workload_name}"
#     )
#     print("#" * 100)

#     # -------------------------------------------------------------------------
#     # Change CustomYCSBWorkload.h for this subrun.
#     # -------------------------------------------------------------------------
#     set_local_ycsb_workload(workload_name)

#     # -------------------------------------------------------------------------
#     # First subrun:
#     #   - commit/push changed workload
#     #   - git pull on every VM
#     #   - compile shab_client + stellar-core on every VM
#     #
#     # Later subruns:
#     #   - copy changed header to VMs
#     #   - compile only shab_client on client VM(s)
#     # -------------------------------------------------------------------------
#     first_subrun = (workload_idx == 0)

#     if first_subrun:
#         subprocess.call(
#             f'git add .; git commit -m "read ratio workload {workload_name}" || true; git push',
#             shell=True,
#         )

#         # Optional local build, preserved from your earlier script.
#         subprocess.call("make -j8", shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         results = run_parallel(
#             git_pull_stellar,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)

#         results = run_parallel(
#             compile_stellar_and_client,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)
#     else:
#         # No git pull and no stellar-core compilation after the first subrun.
#         # The server binary is unchanged. The client workload changes through
#         # CustomYCSBWorkload.h, so update source and recompile shab_client.
#         results = run_parallel(
#             copy_ycsb_header_to_instance,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path("../stellar-private")
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
#         shell=True,
#     )

#     subprocess.call(
#         "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
#         "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
#         shell=True,
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True,
#     )

#     # -------------------------------------------------------------------------
#     # Fixed client load loop.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = (
#             active_clients * client_threads * client_max_in_flight
#         )

#         active_client_indices = [
#             NUM_NODES + j for j in range(active_clients)
#         ]

#         # Compile the client every subrun. On the first subrun it was already
#         # compiled by compile_stellar_and_client(), but compiling it again is
#         # cheap and guarantees the workload header is reflected in shab_client.
#         results = run_parallel(
#             compile_client_only,
#             active_client_indices,
#             max_workers=active_clients,
#         )
#         print(results)

#         run_label = (
#             f"read_ratio_{read_ratio}_"
#             f"{workload_name}_"
#             f"nodes_{NUM_NODES}_"
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(
#             f"🚀 Starting read-ratio run: "
#             f"num_nodes={NUM_NODES}, read_ratio={read_ratio}%, "
#             f"workload={workload_name}, {run_label}"
#         )
#         print("=" * 80)

#         results = run_parallel(
#             clean_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         results = run_parallel(
#             copy_folder_to_instance,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(NUM_NODES),
#             max_workers=min(48, NUM_NODES),
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - NUM_NODES

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients,
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial write batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"SHABDIZ_vs_read_ratio_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={NUM_NODES}\n")
#             f.write(f"read_ratio_percent={read_ratio}\n")
#             f.write(f"workload={workload_name}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"client_wait_after_start_sec={CLIENT_WAIT_AFTER_START_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")
#             f.write(f"compile_stellar_core_this_subrun={first_subrun}\n")
#             f.write("client_compiled_this_subrun=true\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}" """

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, NUM_NODES)) to range(NUM_NODES) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range(min(3, NUM_NODES)),
#             max_workers=min(48, max(1, min(3, NUM_NODES))),
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - NUM_NODES

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log" """

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients,
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")


# # =============================================================================
# # Cleanup after all read-ratio subruns
# # =============================================================================

# # if DELETE_ALL_AFTER_FINAL_RUN:
# #     instances = fetch_existing_instances()

# #     print("\n➡ All read-ratio runs complete. Deleting all remaining tsm-sc-* VMs:")
# #     for name, inst_zone in instances:
# #         print(f"  - {name} ({inst_zone})")

# #     if instances:
# #         run_parallel(
# #             delete_instance,
# #             instances,
# #             max_workers=min(32, len(instances)),
# #         )
# #         print(f"\n🧹 Deleted {len(instances)} instance(s).\n")
# #     else:
# #         print("\n✔ No instances to delete.\n")
# # else:
# #     print("\n✔ All read-ratio runs complete. Leaving VMs running.")


#  Memory Exp

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-a']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [4]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 160
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

# #     if do_compile_this_run:
# #         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

# #         n_collection = 100
# #         subprocess.call('make -j8', shell=True)

# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def git_pull_stellar(i):
# #             inst_zone = get_zone_for_instance(i)

# #             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# # cd stellar-core; \
# # git pull"'''

# #             print(command)
# #             output = subprocess.call(command, shell=True)
# #             print(output)
# #             return output

# #         results = run_parallel(
# #             git_pull_stellar,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )
# #         print(results)

# #         # ---------------------------------------------------------------------
# #         # Compile only on the first/largest run.
# #         # Since later runs reuse a subset of these VMs, no recompilation is needed.
# #         # ---------------------------------------------------------------------
# #         def compile_stellar(i):
# #             inst_zone = get_zone_for_instance(i)

# #             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# # cd stellar-core; \
# # g++ -O2 -std=c++17 -pthread \
# # -I/home/tejas/stellar-core/src \
# # /home/tejas/stellar-core/shab_client.cpp \
# # -o /home/tejas/stellar-core/shab_client; \
# # make -j16; \
# # cd; \
# # sudo rm -rf stellar-private"'''

# #             print(command)
# #             output = subprocess.call(command, shell=True)
# #             print(output)
# #             return output

# #         results = run_parallel(
# #             compile_stellar,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )


        
# #         print(results)
# #     else:
# #         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

# #     # -------------------------------------------------------------------------
# #     # Generate stellar-private configs locally using only replica IPs.
# #     # -------------------------------------------------------------------------
# #     stellar_private_path = Path('../stellar-private')
# #     if stellar_private_path.exists():
# #         shutil.rmtree(stellar_private_path)
# #     stellar_private_path.mkdir()

# #     subprocess.call(
# #         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
# #         shell=True
# #     )

# #     subprocess.call(
# #         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
# #         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
# #         shell=True
# #     )

# #     # Enable custom message only on leader.
# #     line_to_add = "SEND_CUSTOM_MESSAGE=true"
# #     target_file = "../stellar-private/node1/stellar-core.cfg"

# #     subprocess.call(
# #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
# #         shell=True
# #     )

# #     # Optional memory profiling on node2.
# #     if num_nodes >= 2:
# #         target_file = "../stellar-private/node2/stellar-core.cfg"
# #         line_to_add = "MEMORY_PROF=true"

# #         subprocess.call(
# #             f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
# #             shell=True
# #         )

# #     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

# #     # -------------------------------------------------------------------------
# #     # Throughput/latency experiment loop.
# #     # Fixed offered load for scalability:
# #     # active_clients=1, client_threads=2, max_in_flight=100.
# #     # Aggregate max in-flight = 200.
# #     # -------------------------------------------------------------------------
# #     for load in LOAD_POINTS:
# #         active_clients = load["active_clients"]
# #         client_threads = load["client_threads"]
# #         client_max_in_flight = load["max_in_flight"]

# #         if active_clients > n_clients:
# #             raise RuntimeError(
# #                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
# #             )

# #         total_client_threads = active_clients * client_threads
# #         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

# #         run_label = (
# #             f"clients_{active_clients}_threads_{client_threads}_"
# #             f"inflight_{client_max_in_flight}_"
# #             f"total_threads_{total_client_threads}_"
# #             f"total_inflight_{aggregate_max_in_flight}"
# #         )

# #         print("\n" + "=" * 80)
# #         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
# #         print("=" * 80)

# #         def clean_stellar_private(i):
# #             inst_zone = get_zone_for_instance(i)

# #             remote_command = f"""\
# # cd /home/tejas; \
# # sudo rm -rf stellar-private; \
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing: {command}")
# #             output = subprocess.call(command, shell=True)
# #             print(f"Return code for tsm-sc-{i:03}: {output}")
# #             return output

# #         results = run_parallel(
# #             clean_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def copy_folder_to_instance(
# #             i,
# #             source_folder="/home/tejas/stellar-private",
# #             destination_path="/home/tejas/stellar-private"
# #         ):
# #             inst_zone = get_zone_for_instance(i)
# #             instance_name = f"tsm-sc-{i:03}"

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

# #             print(f"Executing command for {instance_name}: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Command for {instance_name} finished with exit code: {output}")

# #             return (instance_name, output)

# #         results = run_parallel(
# #             copy_folder_to_instance,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def run_stellar_private(i):
# #             inst_zone = get_zone_for_instance(i)

# #             node_number = i + 1
# #             instance_name = f"tsm-sc-{i:03}"

# #             remote_command = f"""\
# # cd /home/tejas/stellar-private; \
# # nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# # > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Return code for {instance_name}: {output}")
# #             return output

# #         # Kill old processes on all replica and client machines.
# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         # Start consensus replicas.
# #         results = run_parallel(
# #             run_stellar_private,
# #             range(num_nodes),
# #             max_workers=min(48, num_nodes)
# #         )

# #         print(results)
# #         print("All Stellar nodes should be starting in the background.")

# #         # Give nodes time to authenticate and start the client listener.
# #         time.sleep(80)

# #         def run_stellar_client(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"
# #             client_id = i - num_nodes

# #             remote_command = f"""\
# # cd /home/tejas/stellar-private; \
# # nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# # {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# # {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# # > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing client command: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Return code for {instance_name}: {output}")
# #             return output

# #         # Start only the required number of client VMs for this load point.
# #         active_client_indices = [
# #             num_nodes + j for j in range(active_clients)
# #         ]

# #         results = run_parallel(
# #             run_stellar_client,
# #             active_client_indices,
# #             max_workers=active_clients
# #         )

# #         print(results)
# #         print(
# #             f"Started {active_clients} client VM(s), "
# #             f"each with {client_threads} client threads. "
# #             f"Total client threads = {total_client_threads}. "
# #             f"Aggregate max in-flight = {aggregate_max_in_flight}."
# #         )

# #         # Wait for the duration run to produce stable per-second client logs.
# #         # The clients may wait forever on final partial batches, so we kill them after this.
# #         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

# #         # Stop all nodes and clients.
# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         remote_base_folder = "/home/tejas/stellar-private"

# #         local_base_destination = (
# #             "/home/tejas/work/experiments/shabdiz/"
# #             + f"memory_no_cleanup_{num_nodes}_{run_label}"
# #         )

# #         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

# #         # Save run metadata.
# #         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
# #             f.write(f"num_nodes={num_nodes}\n")
# #             f.write(f"active_clients={active_clients}\n")
# #             f.write(f"client_threads_per_vm={client_threads}\n")
# #             f.write(f"total_client_threads={total_client_threads}\n")
# #             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
# #             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
# #             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
# #             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
# #             f.write(f"leader_ip={node1_ip}\n")
# #             f.write(f"machine_type={machine_type}\n")
# #             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

# #         def copy_folder_from_instance(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"

# #             node_number = i + 1
# #             node_folder = f"node{node_number}"

# #             remote_source_path = posixpath.join(remote_base_folder, node_folder)

# #             local_destination_path = Path(local_base_destination) / instance_name
# #             local_destination_path.mkdir(parents=True, exist_ok=True)

# #             remote_source = f"{instance_name}:{remote_source_path}"

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # --recurse "{remote_source}" "{local_destination_path}"'''

# #             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Copy from {instance_name} finished with exit code: {output}")

# #             return (instance_name, output)

# #         # Copy only a few node logs to reduce time.
# #         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
# #         node_copy_results = run_parallel(
# #             copy_folder_from_instance,
# #             range((num_nodes)),
# #             max_workers=min(48, max(1, min(3, num_nodes)))
# #         )

# #         def copy_client_log(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"
# #             client_id = i - num_nodes

# #             remote_source = (
# #                 f"{instance_name}:/home/tejas/stellar-private/"
# #                 f"stellar-client-{client_id}.log"
# #             )

# #             local_destination_path = Path(local_base_destination)
# #             local_destination_path.mkdir(parents=True, exist_ok=True)

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

# #             print(f"Copying client log from {instance_name}...")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Copy finished with exit code: {output}")

# #             return (instance_name, output)

# #         client_copy_results = run_parallel(
# #             copy_client_log,
# #             active_client_indices,
# #             max_workers=active_clients
# #         )

# #         print("\n--- Summary of Download Results ---")
# #         print("Node log copies:", node_copy_results)
# #         print("Client log copies:", client_copy_results)
# #         print(f"Saved run to: {local_base_destination}")



#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j2; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )


        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     if num_nodes >= 2:
#         target_file = "../stellar-private/node2/stellar-core.cfg"
#         line_to_add = "MEMORY_PROF=true"

#         subprocess.call(
#             f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#             shell=True
#         )

#     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"memory_withv3_cleanup_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

# PBFT Test

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-a']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [4]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 160
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output




#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j2; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )


#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output



#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(40)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"SCP_test_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

# SCP

In [ ]:
# import subprocess
# import concurrent.futures
# import shutil
# import time
# import re
# from pathlib import Path
# from datetime import datetime


# # =============================================================================
# # Basic helpers
# # =============================================================================

# def run_shell(command, check=False):
#     print(f"\n$ {command}")
#     rc = subprocess.call(command, shell=True)
#     if check and rc != 0:
#         raise RuntimeError(f"Command failed with code {rc}: {command}")
#     return rc


# def run_shell_output(command):
#     print(f"\n$ {command}")
#     return subprocess.check_output(command, shell=True).decode()


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # =============================================================================
# # Experiment config
# # =============================================================================

# PROJECT = "research-488322"
# ZONE = "us-central1-b"
# MACHINE_TYPE = "e2-standard-2"
# IMAGE_FAMILY = "tsm-sc-family"
# SUBNET = "default"
# GCP_USERNAME = "tejas"

# # Run in decreasing order so we can delete unused nodes after each subrun.
# NUM_NODES_LIST = [48, 32, 16, 8, 4]
# MAX_NUM_NODES = max(NUM_NODES_LIST)

# N_CLIENTS = 0

# # You said NUM_TEST_ACCOUNTS is already set to 100 in C++.
# NUM_TEST_ACCOUNTS = 100

# # This is only used for summary labels. It is not patched by this script.
# SCP_OPS_PER_TX = 100

# RUN_DURATION_SEC = 210
# MEASURE_START_SEC = 120
# MEASURE_END_SEC = 180

# LOCAL_PRIVATE_DIR = Path("../stellar-private")
# RESULTS_DIR = Path("/home/tejas/work/experiments/stellar_native")
# RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# DELETE_BEFORE_FIRST_RUN = False

# # Final cleanup deletes whatever remains after the 4-node run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # Compile only during the first subrun.
# COMPILE_ONLY_FIRST_SUBRUN = True


# # =============================================================================
# # GCP helpers
# # =============================================================================

# def instance_name(i):
#     return f"tsm-sc-{i:03}"


# def get_zone_for_instance(i):
#     return ZONE


# def fetch_existing_instances():
#     cmd = f"""
#     gcloud compute instances list \
#         --project={PROJECT} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     """
#     output = subprocess.check_output(cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     return instances


# def delete_instance(instance):
#     name, inst_zone = instance
#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={PROJECT} \
#         --quiet
#     """
#     print(f"Deleting {name} in {inst_zone}")
#     return subprocess.call(cmd, shell=True)


# def delete_instance_by_index(i):
#     name = instance_name(i)
#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={PROJECT} \
#         --quiet
#     """
#     print(f"Deleting unused node {name}")
#     return subprocess.call(cmd, shell=True)


# def delete_nodes_not_needed_next(current_num_nodes, next_num_nodes):
#     """
#     Because NUM_NODES_LIST is descending:
#       after 48-node run, delete 32..47
#       after 32-node run, delete 16..31
#       after 16-node run, delete 8..15
#       after 8-node run, delete 4..7
#     """
#     if next_num_nodes is None:
#         return

#     to_delete = list(range(next_num_nodes, current_num_nodes))

#     if not to_delete:
#         return

#     print(
#         f"Deleting nodes no longer needed for future subruns: "
#         f"{instance_name(to_delete[0])} to {instance_name(to_delete[-1])}"
#     )

#     run_parallel(
#         delete_instance_by_index,
#         to_delete,
#         max_workers=min(32, len(to_delete)),
#     )


# def create_instance(i):
#     name = instance_name(i)
#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances create {name} \
#         --project={PROJECT} \
#         --zone={inst_zone} \
#         --machine-type={MACHINE_TYPE} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={SUBNET} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={IMAGE_FAMILY},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     print(f"Creating {name}")
#     return subprocess.call(cmd, shell=True)


# def ensure_instances():
#     existing = fetch_existing_instances()
#     existing_names = {name for name, _ in existing}

#     needed_indices = list(range(MAX_NUM_NODES + N_CLIENTS))
#     missing = [i for i in needed_indices if instance_name(i) not in existing_names]

#     if not missing:
#         print("All required instances already exist.")
#         return

#     print("Missing instances:", [instance_name(i) for i in missing])
#     run_parallel(create_instance, missing, max_workers=min(32, len(missing)))

#     print("Waiting after VM creation...")
#     time.sleep(30)


# def get_node_records(num_nodes):
#     cmd = f"""
#     gcloud compute instances list \
#         --project={PROJECT} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     """
#     output = subprocess.check_output(cmd, shell=True).decode().strip()

#     records = []
#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             records.append((idx, name, inst_zone, ip))

#     records.sort()
#     node_records = [r for r in records if r[0] < num_nodes]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} nodes, found {len(node_records)}: {node_records}"
#         )

#     return node_records


# def kill_stellar(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas; \
# sudo pkill -9 stellar-core || true; \
# sudo pkill -9 shab_client || true; \
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def git_pull_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas/stellar-core; \
# git pull
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def compile_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas/stellar-core; \
# make -j2; \
# cd /home/tejas; \
# sudo rm -rf stellar-private
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def clean_remote_private(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def copy_private_to_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     cmd = f"""gcloud compute scp --zone "{inst_zone}" --project "{PROJECT}" \
# --recurse "/home/tejas/stellar-private" "{name}:/home/tejas/stellar-private"
# """
#     return subprocess.call(cmd, shell=True)


# def run_stellar_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)
#     node_number = i + 1

#     remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def collect_node_log(i, local_run_dir):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)
#     node_number = i + 1

#     local_run_dir.mkdir(parents=True, exist_ok=True)
#     local_path = local_run_dir / f"{name}-node{node_number}-stellar-core.log"

#     cmd = f"""gcloud compute scp --zone "{inst_zone}" --project "{PROJECT}" \
# "{name}:/home/tejas/stellar-private/node{node_number}/stellar-core.log" \
# "{local_path}"
# """
#     return subprocess.call(cmd, shell=True)


# # =============================================================================
# # Config generation
# # =============================================================================

# def write_tsm_ips(node_ips):
#     with open("tsm_ips.txt", "w") as f:
#         for ip in node_ips:
#             f.write(ip + "\n")


# def generate_stellar_private_configs(num_nodes):
#     if LOCAL_PRIVATE_DIR.exists():
#         shutil.rmtree(LOCAL_PRIVATE_DIR)

#     LOCAL_PRIVATE_DIR.mkdir(parents=True)

#     run_shell(
#         "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
#         check=True,
#     )

#     run_shell(
#         "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
#         "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
#         check=True,
#     )

#     # Only node1 injects native Stellar workload.
#     # ENABLE_SCP_TRACKING is already true in C++.
#     for node_number in range(1, num_nodes + 1):
#         cfg = LOCAL_PRIVATE_DIR / f"node{node_number}" / "stellar-core.cfg"

#         if not cfg.exists():
#             raise FileNotFoundError(f"Missing config: {cfg}")

#         lines = cfg.read_text().splitlines()
#         lines = [
#             line for line in lines
#             if not line.strip().startswith("SEND_CUSTOM_MESSAGE=")
#         ]

#         if node_number == 1:
#             lines.insert(0, "SEND_CUSTOM_MESSAGE=true")
#         else:
#             lines.insert(0, "SEND_CUSTOM_MESSAGE=false")

#         cfg.write_text("\n".join(lines) + "\n")


# # =============================================================================
# # Log parsing
# # =============================================================================

# def parse_ts(line):
#     # Example:
#     # 2026-06-30T00:09:11.842 GDQOM [Ledger INFO] Got consensus...
#     m = re.match(r"^(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?)", line)
#     if not m:
#         return None

#     try:
#         return datetime.fromisoformat(m.group(1))
#     except ValueError:
#         return None


# def parse_committed_ops(log_path, start_sec=MEASURE_START_SEC, end_sec=MEASURE_END_SEC):
#     entries = []

#     with open(log_path, "r", errors="ignore") as f:
#         for line in f:
#             if "Got consensus:" not in line:
#                 continue

#             ts = parse_ts(line)
#             if ts is None:
#                 continue

#             m_ops = re.search(r"ops=(\d+)", line)
#             m_txs = re.search(r"txs=(\d+)", line)

#             if not m_ops:
#                 continue

#             ops = int(m_ops.group(1))
#             txs = int(m_txs.group(1)) if m_txs else 0

#             entries.append((ts, txs, ops, line.rstrip()))

#     if not entries:
#         return {
#             "ops": 0,
#             "txs": 0,
#             "throughput_ops_s": 0.0,
#             "avg_ops_per_ledger": 0.0,
#             "avg_txs_per_ledger": 0.0,
#             "min_ops_per_ledger": 0,
#             "max_ops_per_ledger": 0,
#             "min_txs_per_ledger": 0,
#             "max_txs_per_ledger": 0,
#             "num_ledgers": 0,
#             "first_ts": None,
#             "last_ts": None,
#         }

#     t0 = entries[0][0]
#     selected = []

#     for ts, txs, ops, line in entries:
#         elapsed = (ts - t0).total_seconds()
#         if start_sec <= elapsed <= end_sec:
#             selected.append((ts, txs, ops, line))

#     total_ops = sum(ops for _, _, ops, _ in selected)
#     total_txs = sum(txs for _, txs, _, _ in selected)
#     num_ledgers = len(selected)
#     window = max(1.0, end_sec - start_sec)

#     ops_values = [ops for _, _, ops, _ in selected]
#     txs_values = [txs for _, txs, _, _ in selected]

#     return {
#         "ops": total_ops,
#         "txs": total_txs,
#         "throughput_ops_s": total_ops / window,
#         "avg_ops_per_ledger": total_ops / num_ledgers if num_ledgers else 0.0,
#         "avg_txs_per_ledger": total_txs / num_ledgers if num_ledgers else 0.0,
#         "min_ops_per_ledger": min(ops_values) if ops_values else 0,
#         "max_ops_per_ledger": max(ops_values) if ops_values else 0,
#         "min_txs_per_ledger": min(txs_values) if txs_values else 0,
#         "max_txs_per_ledger": max(txs_values) if txs_values else 0,
#         "num_ledgers": num_ledgers,
#         "first_ts": selected[0][0].isoformat() if selected else None,
#         "last_ts": selected[-1][0].isoformat() if selected else None,
#     }


# def write_summary(run_dir, num_nodes, leader_log):
#     summary = parse_committed_ops(leader_log)

#     summary_path = run_dir / "summary.txt"

#     with open(summary_path, "w") as f:
#         f.write(f"num_nodes={num_nodes}\n")
#         f.write(f"num_accounts={NUM_TEST_ACCOUNTS}\n")
#         f.write(f"ops_per_tx={SCP_OPS_PER_TX}\n")
#         f.write(f"measurement_window={MEASURE_START_SEC}-{MEASURE_END_SEC}s\n")
#         f.write(f"committed_ops={summary['ops']}\n")
#         f.write(f"committed_txs={summary['txs']}\n")
#         f.write(f"throughput_ops_s={summary['throughput_ops_s']:.2f}\n")
#         f.write(f"avg_ops_per_ledger={summary['avg_ops_per_ledger']:.2f}\n")
#         f.write(f"avg_txs_per_ledger={summary['avg_txs_per_ledger']:.2f}\n")
#         f.write(f"min_ops_per_ledger={summary['min_ops_per_ledger']}\n")
#         f.write(f"max_ops_per_ledger={summary['max_ops_per_ledger']}\n")
#         f.write(f"min_txs_per_ledger={summary['min_txs_per_ledger']}\n")
#         f.write(f"max_txs_per_ledger={summary['max_txs_per_ledger']}\n")
#         f.write(f"num_ledgers={summary['num_ledgers']}\n")
#         f.write(f"first_ts={summary['first_ts']}\n")
#         f.write(f"last_ts={summary['last_ts']}\n")

#     print("\n" + "=" * 80)
#     print(f"RESULT num_nodes={num_nodes}")
#     print(f"committed_ops={summary['ops']}")
#     print(f"committed_txs={summary['txs']}")
#     print(f"throughput_ops_s={summary['throughput_ops_s']:.2f}")
#     print(f"avg_ops_per_ledger={summary['avg_ops_per_ledger']:.2f}")
#     print(f"avg_txs_per_ledger={summary['avg_txs_per_ledger']:.2f}")
#     print(f"min_ops_per_ledger={summary['min_ops_per_ledger']}")
#     print(f"max_ops_per_ledger={summary['max_ops_per_ledger']}")
#     print(f"num_ledgers={summary['num_ledgers']}")
#     print(f"summary={summary_path}")
#     print("=" * 80 + "\n")

#     return summary


# import os
# os.system('git add .; git commit -m "update"; git push')

# # =============================================================================
# # Main experiment
# # =============================================================================

# if DELETE_BEFORE_FIRST_RUN:
#     instances = fetch_existing_instances()
#     if instances:
#         run_parallel(delete_instance, instances, max_workers=min(32, len(instances)))

# ensure_instances()

# max_node_records = get_node_records(MAX_NUM_NODES)

# print("All available node records:")
# for r in max_node_records:
#     print(r)

# all_results = []

# os.system('git add .; git commit -m "update"; git push')

# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
#     run_name = f"stellar_native_nodes_{num_nodes}_accounts_{NUM_TEST_ACCOUNTS}"
#     run_dir = RESULTS_DIR / run_name
#     run_dir.mkdir(parents=True, exist_ok=True)

#     print("\n" + "#" * 100)
#     print(
#         f"Starting Stellar native run: "
#         f"nodes={num_nodes}, accounts={NUM_TEST_ACCOUNTS}, opsPerTx={SCP_OPS_PER_TX}"
#     )
#     print("#" * 100)

#     active_indices = list(range(num_nodes))
#     active_node_records = get_node_records(num_nodes)
#     active_node_ips = [r[3] for r in active_node_records]

#     print("Active node records:")
#     for r in active_node_records:
#         print(r)

#     print("Active node IPs:", active_node_ips)

#     # Kill all currently-existing nodes before each subrun.
#     # For first run this is 48; after deletion it becomes smaller.
#     existing_before_run = fetch_existing_instances()
#     existing_indices = []
#     for name, _ in existing_before_run:
#         if name.startswith("tsm-sc-"):
#             try:
#                 idx = int(name.rsplit("-", 1)[1])
#                 existing_indices.append(idx)
#             except ValueError:
#                 pass

#     if existing_indices:
#         print("Killing Stellar on all existing tsm-sc nodes before subrun...")
#         run_parallel(
#             kill_stellar,
#             sorted(existing_indices),
#             max_workers=min(48, len(existing_indices)),
#         )

#     # Compile only once, during the first subrun.
#     if (not COMPILE_ONLY_FIRST_SUBRUN) or (run_idx == 0):
#         print("Compiling Stellar Core on all max nodes. This happens only for first subrun.")

#         run_parallel(
#             git_pull_node,
#             range(MAX_NUM_NODES),
#             max_workers=min(48, MAX_NUM_NODES),
#         )

#         run_parallel(
#             compile_node,
#             range(MAX_NUM_NODES),
#             max_workers=min(48, MAX_NUM_NODES),
#         )
#     else:
#         print("Skipping compile for this subrun.")


#     print("Stopping active nodes...")
#     run_parallel(
#         kill_stellar,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     # Generate configs for the current node count.
#     # This rewrites tsm_ips.txt so the generated private network has exactly num_nodes validators.
#     write_tsm_ips(active_node_ips)
#     generate_stellar_private_configs(num_nodes)

#     # Clean and copy configs only to active nodes.
#     run_parallel(
#         clean_remote_private,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     run_parallel(
#         copy_private_to_node,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     # Start only active nodes.
#     run_parallel(
#         run_stellar_node,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     print(f"Nodes started. Running for {RUN_DURATION_SEC} seconds...")
#     time.sleep(RUN_DURATION_SEC)

#     print("Stopping active nodes...")
#     run_parallel(
#         kill_stellar,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     print("Collecting logs...")
#     run_parallel(
#         lambda i: collect_node_log(i, run_dir),
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     leader_log = run_dir / "tsm-sc-000-node1-stellar-core.log"

#     if leader_log.exists():
#         result = write_summary(run_dir, num_nodes, leader_log)
#         result["num_nodes"] = num_nodes
#         all_results.append(result)
#     else:
#         print(f"WARNING: leader log not found: {leader_log}")

#     # Delete nodes that are no longer needed for the next smaller subrun.
#     if run_idx + 1 < len(NUM_NODES_LIST):
#         next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#         delete_nodes_not_needed_next(num_nodes, next_num_nodes)


# # =============================================================================
# # Final CSV summary
# # =============================================================================

# csv_path = RESULTS_DIR / f"stellar_native_accounts_{NUM_TEST_ACCOUNTS}_nodes_sweep_summary.csv"

# with open(csv_path, "w") as f:
#     f.write(
#         "num_nodes,num_accounts,committed_ops,committed_txs,throughput_ops_s,"
#         "avg_ops_per_ledger,avg_txs_per_ledger,"
#         "min_ops_per_ledger,max_ops_per_ledger,"
#         "min_txs_per_ledger,max_txs_per_ledger,"
#         "num_ledgers,first_ts,last_ts\n"
#     )

#     for r in all_results:
#         f.write(
#             f"{r['num_nodes']},"
#             f"{NUM_TEST_ACCOUNTS},"
#             f"{r['ops']},"
#             f"{r['txs']},"
#             f"{r['throughput_ops_s']:.2f},"
#             f"{r['avg_ops_per_ledger']:.2f},"
#             f"{r['avg_txs_per_ledger']:.2f},"
#             f"{r['min_ops_per_ledger']},"
#             f"{r['max_ops_per_ledger']},"
#             f"{r['min_txs_per_ledger']},"
#             f"{r['max_txs_per_ledger']},"
#             f"{r['num_ledgers']},"
#             f"{r['first_ts']},"
#             f"{r['last_ts']}\n"
#         )

# print(f"\nFinal CSV summary: {csv_path}")

# if DELETE_ALL_AFTER_FINAL_RUN:
#     print("Deleting remaining tsm-sc instances after final run...")
#     instances = fetch_existing_instances()
#     tsm_instances = [
#         (name, zone)
#         for name, zone in instances
#         if name.startswith("tsm-sc-")
#     ]

#     run_parallel(
#         delete_instance,
#         tsm_instances,
#         max_workers=min(32, len(tsm_instances)),
#     )

In [ ]:

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")